# CDK journal extension: reproduce and check every result

This notebook verifies the results of the journal extension of the CDK paper
on Google Colab. It carries its own copy of the source code and of every
result file, so Section A reproduces each reported number from the released
data, and Section B re-runs the algorithms from scratch on demonstration
cases so the behaviour can be verified live.

Files embedded: the conference source (`cdk.py`, `baselines.py`,
`datasets.py`), the extension modules (`cdk_geom.py` = certified candidate
family, `cdk_hull.py` = convex-hull reference, `cdk_robust.py` = trimmed
test, `uniforce.py`, `wbp_baseline.py`), the conference result files, and
the six new result files produced for the extension.

Runtime note: Section A is seconds; Section B takes a few minutes on a free
Colab CPU.

In [ ]:
#@title Setup (installs diptest for the UniForCE demo)
!pip -q install diptest
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 40)

In [ ]:
#@title Embedded source and results (click the arrow to inspect)
import base64, gzip, json, os, sys

PAYLOAD = r'''{"src": {"cdk.py": "H4sIAFYSmWoC/7Vca3PbNtb+rl+B1513SiaSYjlJt+tUndqO2maaOB3bzXir0WggEpIYUSTDi2xlZ//7PucAIEFJTpttk9ldiyBwAJz7jXt0dNS5UHkZzSMVipfRJiqijRK/9NZKJoXwLl7+4nfFRuVFlCbipN8Zfsa/zqt1Fqu1SspClEsl5rlcq7s0X4l0Lq5VUAJmIQZPe4NnNEJTcpWledkVd1G5FOVdKoI0z83MTrmUpbhTOSCleYDzzrYabpQXpcjTKgkJjrrPVB7pbSWGJBaEaVDRCBYtAaCTSyzMsVomYhmFoUr6nc7F26ur0cXNq7eXYiC8jYyjMCq3Yq0CTIuKtd/v3NRn/LoQF2nyHmerAH/wtD8QWZ5maYEtAiydYYsoWfABkzRUIusBYqWEXMgoKcrOkRRVEuEm63grchw0XQN0LGcqjmmhwUiWRnQRPF0c9YW4IRwEwEaZV4wWEeGWnSTtpdmpkAJXX1el5DdYw6sFAy2ABFXgLFiP+ywUg19XcRkVCignVClQettxYPSiZCPzSAJIEFdFCcTiaDJepDkoRAfG9RNN3gLUFTMA43PxWQnRdLGo0CdaAwTNzRXuDVQEpcCoFFejH0dXo8uLkbg+e/Pr6xEYYBkFS6G5UIqkimOxBhbjDq5bJXKTRqGcxUrvwghRHyqVBIrwwYxyFEZAUjSr+B7zXKkjIbMsjoCEMqVTdK5vzm5eXd+8uuDLE2b0C3H52+vXBBoMSFsXzEMg7yYKQd5CTwpSXEZTqSOLolpnfMcALDVTdPQCnBGedgT+EZTh14beX2PA/BTphtlQiV8vznpgm0VCfE2sTIiepfeWD4iHugxr7x+9jrFd2ZsDMVVOiKENgDAZi2IpM+UcYiGrogBFcYpLjxDcFdfRYi17QBpkPeoDqQQRgxdEcvFE/GTUQXPLlrCcCE/mwTIqtTBATM5Ehh/g6zLNemF6lxDyILEkKkycwOicQrOcKECXEmhjtghVESiNgKUkju1kKcReyFLriDRlvEuxlsm2Z/hShLKUzMgzFeCKylENpcL/sJwclSoRC6iKrGCtVoEX6pEjkHSdgd0LEqvE8GCiItYVBWjPQPArWcSqYzYGn7wxtCZmPjmGiorjXqEASZLC+ckgXEtJjRErV1qEVNhZDQeGnV1san62EvP23eiq9+vZ1c0rxvzNz6NLcTHC44//Eudvb27evun99uupmFVRHHbAiFUC0YAWA8nACkY4Ca+RymmXreCzgO0lDlKkMUuLmMdpmndp38SQattREsiANlE5gWKVNs+hswz3bUDLKruTedgF7+cL2uWOzq25lwQ4Yu3TyRVpTRXyZQmhMQRqnRIAViOkeYoyghIFJqPFknSEkcs3o6ufRuCPICJ7ZLREx5gS8W3biggQLDFcJjXsOyVXxAtaLZrZd8sU8jKX6yjewgpoFd9GhUim6ygxpBDFWsYxgWG20VSlR5L8UBHdWJ/o2Z00gRx8qEBqMibM/TO1ZHzF0Uq1EU88swaI/IWImGX1VcBFBLG4U1nZsl2gbqBkDGR2jmDKmSDT6bwizplORbRmRMgEeJfaiJqhBIK8JbQkmV5F4hPEkHCcy8yph7qQIwWG4onFCtTOk76VOzP5lzekIsyUIMq2/SLDljLukx6WrJn1zIAGuiKjP51O5yvR+xv/ARxJUQHuUY7xUglYctvwFrShN/rQF/9os4z/d58mVHOz9/TGO+uK8y6RGGw5JY06JRT4xkLgbXIuhpClxDuDIqa/5z6/i+Z4Lb6DoiVWPLe/diGJ74bi+LS2EVqxiOP+MQ/JGYAz7vkgfp/WeXoDKfEu0+9aL2az+kV7hQEO6ZCl5530j8Uj2qFHoHpY5sNs7N0U1CaE8ABrEpbDqWUQ77YLGcq6Ik8WFisGIbcNIuj2D16SJ3xPUJopt4BxOwbMfrBMI2yT2G1UFstADX+EBlL+ZP9e+uK39uL+l+DWE3CrdZLI9Fv/SXjwY9OENBgUYe2DdgWx7bdfhFGncLiZWgVRAgiDNwAiFcOTY0MO6JgzEUbkkYsT4xSwq77SD48fi0IpstxQpLQAmmAVlT3WGPCYtZIA9vOctHJBxp9uF8Hlxb1hqtjTnUf3ZEzxcqlkSD4tA4NMx9YzJHwlMK8FGZnE+AXQoQkbTAh+umJ7DLWD5eINTIjqXcg8TkWYyzuGx67nXN0B2EfA0J72i9pzI3/j2T/EumAIMMAwZgSP1netGw/OGzzHHAZI5s6Y8CiB1AvYjqxtwO/gN5PHgbMaRMLokOo+xO3HeCa+JVlZgBBeol+E9MLzbiFpt+PoeOKLR4/Eid+Hd+YN9JQSPtIQE3nMgBtgBKC8tihkQ4B7Qgt8n+SHVn4vjgUiBsXTPZzjsRj44v+F2f8CgJKsv4HYBitvzGfo0lEGk4meEbO2wZyPKk8L2iYst5kaYgQgv3mmZwEjYkqeVk7xiMfM5jeCG2oQCi7n1iMYJ77fvByfdsXxpEbExfggHuqpA3fq4KGpCbgB0+yK7+w2fh/uNS7gtS9gVM5Uxw9ZH3wtt1NEIjL2AKpLeHAu5GBG3bVGZzk8k87DcwhT7wlT3jEOtgNyTWxDS4bifesFjrbuw0f2dhYwDcfvJ6wY1xOt3o4Nbz2E9D+J8D+JbAgq3EoyO1rVhmNGH/NB4jPiJpp3W/YGw1271liTKdyuENEgdAGprdeu+SApG4rx5BC3vXaQwmA/YOqeCmzoXBjpwVz3YAbPx+CVAv/V4ttGN07RR+SJkMbz9E7tS+G9uQuplmltAsx1SlkZbawtqYEe0HEP3r22hBTq75nKSxgVo5vhIwHIWt57+tiw4XoLphfIqHpPjw1fxAtM7Vke98YfND4JbUBpMCGDryE+YYh8agAYnNTre2DTRZ9e6qHMsNp95sUL/wWen8DZcDSWOXEw3tFadNrAJ9WV+f4EfPglDPNTMsy10vcogvEpx5Eh9PsCxneNyGRa70fUXEVJ2NjdK40LKYI4Leo4mhiGQ1JOB1HMrYrCObY+r3VyKUOjSR99VFDJlCTjxB4HF0Q8Y70ohZZwJhCxFmLitIhsrikE+9R2VVstCgXdoORBq9sYui7rmdt+k5xYVzxgVNELcUv8TQpkXVl+JoSQkjsyyZOjhrfBh/jPu1JzVAzXAJxWbELvNkDwAtJN17LMwTuF8fbqlb/TNoH4AYv7N41CSLuIkfHq9z78DByoS7/AuZCGxq4YifUahbMja8Sz5rCeBtll3A9/1zf3eV9YV3NJZ2ntpbgXt7kb5+bGFsuSM0DTk5BMVJBu+OZ5eocIyFzZJysOgSRXncRuq7ywwUOZb9tXeB20kBlQlFystt5Fs0bdBwhInUmvo+QsXozyPM3bwGAK37XAUWDvgqp3fAeUYFoYyQXdpPiQl3yjOMo8QIH5IwXm+59NB4oxQpmH0wTUgH1mp8QnCrwO+jefpkEuoZPFO8rf8uU8lk2jtGtpm2pps7LrGiID8AEh54N/CR32DPL8Y5pTVua6hCv6RFyXeZos+AEh8E9fX6s4FqoUiNPFyfHgmy8SWMz1EaYFtvUysn+nlrVJlWc+MfeByNW1EswAxOiFNkAAw54DJBPBH0wNVIVjboJqbdZVazIoZLnidOENEKn2BMwGcNF4HIOuPQg5utp1WOn1nMPyCByi6w8+GRvniGSY0tW4B4eHXWRzo3Tla//52PDITCXvJTRJNF2mwXKm8oWLB6jFV7Dj0E5NOkLnyMRym6UYoKxMlYTQqec/U1wSI3KKYXq9CtyMYczUCSy/1rEGb3v46tQuI9/YEmJ9mAaNwZ+Q42WWpwZ2vihSyKeBUi5zjH+AevFaqF0zbgjja30yWVAK1cVvNk4nhGCAqFFsuUPP9j/nfNaxGp/azSyJJoYgBcvBl+JIZsQ9nmyQ/kLEBgjxpEHfBlra4pVvZHyjRLtEcTaOTtnN8taAGwGh5IFFjUe79iefIMRaM3dNhTaD8+Y7BPgMHv/bNdfzvq5r1KWmv3uLH+rsZof/V1y8/OVKFVVcaqqvTuniNpjFRU8JWWRBQBvNBoSzUxFT5m2oM6QeWEsCxHQugzLNt8OY016skNJkPi1UeSrKKmO/m0TDKJtkSgnkgrfEG81mqyn0RWnHBjqGCfMpy/6p5rU6fjpK4BVoYIwoXeoK5CyKKe1M5jRJOfP5GTjS0DIERFhfp7/Nk61n1hUsnbVR4cIm03VNUeexKROuwRXRggoI7L5mUFxRUJokDihgCj7lkkunqUmnn3Jeot7aAJLhexkoKm8qOYecRzll4CnomVGZJQ51fcJUD3XWhoAQ4bmgZwOhr4g4ZbSo0qqAiS4of6T95YrrhSH2JeeRr7OachU6AsggjWOZUaUpJfNWiEHf3lFlU0XFQg4d5rkMbEGW0FMY9c5ajXP8ASnxEMwOnaTrTLhERYgzAC/f3nChII64uGfwRjUat9plS0liBcRRvFlQjqtB/cXby0uq2b17dfOvOiao8k20IXRT1UaJRS6zZZdjR70W16x0scScOk2KU0sC7JNg82e9y0uX8lwKgcO1Jcwm2pJBs6TVYomrEYWTXo0DAwvWkSR9a0IQTFfFkqhIZRGHBW1xnXhtnUU5FYqwI0h2rYL+u1e9CwPQDUm4cklVEo5+HOo0cmRTyHavT0iYla0Yy2dpGmPOTV4p6MFalWgMcX6CxXg69QoVz2FphtjoeVdcDQf//CcH92bg9fBpV5do6/jmQL2XKhjDI11SOOrqutTwBMuLajYNZDY8OYYzRDU7fnrKT7qHYkhnPAAyr2I1PJotAW6FEGf4zTOCm8yG+ItnrZqG3wDS/lp6n2l84Ro4hk5lT+mYanjseON0+/6Hrv57Zf5ykoB/vSaDBayYdMfr9jpCi5lIkM1PXZQbCvOWX/BYe7G5PlU/9K/2a4M4A9NiDrPrF3asvY7QZhYR2ihVyyP0sHN6INPMrPFJx+bheqS9xkGsTtHYp51DOOimEziPncYY9JzC1/9qL2t2vjGMfNvVaTOdMXKDHfpHJSZOL5rEpE4Qm4fBxE2d6YKXLWrpsle7yHOg0GPW1izB4bERi9PdJKjZ4fsWwfdTomc48Nl4J89Epbg2nxwqG+1sdv7HmxF2znc3O/+szQxKduqLJkn4MJZkuIMht2aLSUVTNkZkUaRJa/KGkGTTNPBDz5uUTbJphfgUZ3sbfxc/mPWd9or3kXKAyPTvI3lLt4jUNxy2bfZffsRJPtoyHbnIH3GV0LMpj5P2KfZyHXuVP3t172NXHNFFjvx+LUNtYCYJMuI/MI5/6lZ7CYWaRH5LbinZZnplAt1CxZ0G/4PYmjaOlvBO01lhUsvsVA45tdJcAKHkz8enO21ot9Z30u0PMNdXThOY8Qr7TXaGjDn2UfmGWqdqPUSWPF1nVanbZmjt2/Pr0dW70UsqHJeR9ZhuyQFOG3amZkJqDKxyqi6Tlu7p9h3jODjJz4ObzbY1LLLjW9ukd7AIW9dddYsMe5IP5FRtXlUnUym717WdYsC5gafmczJFOA9cuILOj8EZ/F1uLyoctJ3Xg4xvRWiVBllSd4gRPqtZ+zicjPWsxfIbk03+r0nN0gFzBacxNCc0jVU1lWhCeRiVjZmiU9VINRC4BdAciHDRd1lpV+ff1krSHrctOnhl6p+Ohrz1dwz1ropsgdClf8xvFKcpcAw/2YWwo4PdMlA0tyDa3RaOoA8cQf/qIf7nVsgZdxWGldPMen32ZuSwnWxCJ+PVHqBJV1COKL+LTBxA8kHkISce6om7PU1TYKiDNQcg87SlOGsXiuiMe1643W22Xn0KY5HmVm4W0UYlDrxGeG2JXMLCwJeHk66ZsLlCI2h1wst2izkQKc1BV1rKeKNTZLYHr+6tazEgBQeEOLPOgbRQSRUl1JjYtOeZPiarPmpSNWdzC/gOMH0ZQ7YWxagBUAe/Px/rAE+GNH5EM+Hca8XGaT0HHl3RtCZGRaNgj/pumXJKqbf9MqH2oV2v2i0YtiSPYWAD0vV/yL4lmQjyhQkuXAxrN9puXz19GbFnyzUim8bg5IeKYp3xoxrKc+4WsgHBcf/kue+gYXRPbcFK5kSlMs0MXebkVNRJB5zfGzym7fwn+HHl276QgOtP8dYBGCUBCFBolqhyDgr5oNSFstFamvViGyIbdhXuyp9tPjQpWNabVQlFiPVhyiKnedm00nJUzWG+jIkPti1YYDzEKy2Vu4bQEK7aDSfaCtzJgliWeugdrthwXpFOw5qVf+hGb6KxTtJpbNdrIAhMjN1yRB1t+a3WA6dsruO3nTLLLeUbAdNrcQOr20+r2vyTupZvlx9g+Pxhjg93XL1prmN7xk1uGd9BCnE14MU5LT5wAGaVx7r2T9C+H2qpgMPLfP7Y5uYcIeMXmHjynAnLVYfHDImdU/1Ik8gEGuHY9xt1U8huDvoTwNquI2nxRhn/lZBPt/c2rmM71INp/+1PtxmXB7prG1Z2PUnOq6Zznd2LYLjuBedeIfklN1hF3BnXvqbfb/mdSbWeQdo1FLIdIP/L0dUrcjI56kkcwTvU8vvC8TutUEkB1ynvwSkpqJGL9ne+jZhV4UKVfdff0Xt3xYdKVYr7UbrCaXfRbo0T20GTwbTx7J1QNqQ0A7/oZ2lmW3Z2/CrM0tH0IzdNYkJrfRaKi+iJQfnEqXUeY58N9RLbwULQX+iEaVKpA61Nu7I6xorJQ/J6+IOGlgJ5UHe4gFv6Yx8rONhBi/dZ15NdQffDDDep4TwODuYCJBFjnxCz9vBfOJdmK6ihMZ1wsqsvNJyWagjqL76MG0Rcz4lf8bmqgfPZTlBpWJ11MSW42nriwrYr6Tz4qeCPCljC36dsB6EiOEGMpSKBE0D+qMGEK1IXrULZ7Tium9jYXsVkr/SyScMNL+sG6IuuuCAZMG0QRI8Ln9qfBmrwbT1/xI5P6bVtoWxsoVm3k03BnBnNcQqkL8dy4o9P6VaTfUqP+jIMPY8aTSQXMcEcPmcGnWeHqw1hC/4wwBt1xUpth7Fcz0Ip1Kl4OVZj4kw15nbMhlbk3D9EMB52cwFtwlm/iy6nv9gJWqSEk2YzBa5/yz5trXprgG7y4Ey8PqOPOuwOuqmYUwlU0TE6t+FYWdS+exOKGh3OFQisWKmsrYEpnfvv/7TISKiFDIJOmhXbokt4n7FbRWcH9SymxrPJAZXMLTdJgDNRT7I31usn/qHOT9sTa1W1Llu/oNdjHpP+6UTsOhXZWB94UjvjNqFj9SAnYx367bJLpjnhBw4bg7Uql2nopIeoy4u6zQsvIXk0HDElVDqsoFv6CTEkRqZdk+bv9uPAXQ+9+x3R0IZNwxjfT8T/DcUBe9O8H9rf9dDkhbhvhu8P5kPvO2157WpxbO6zE/VgQj7jCi6OLP2u/jHbsyK5pAPnswdPnEs6cj7bxfyYAUZ79XqNuElLNZP3/5cK3Z2GAqUVc7/9MYTTs3Db6go50IPO17C9GjpF1rdFbox7e1UIf8ftqfnVOJG7bbW6ym02Nd6JG6/a9xCIg9FqXbb3VsNB19Tqh27nOYlX18DBlE+7HG5tcMiNRbYeOHTLgH8ApSkQ2qxU0zPHBWCLFK2Ndw1nXTVy42L+UpfAUj2/Lrr/+OqWfFlddTX1ZadpiLOzps4qpJum4Yn0uVdeFk0K0NHXFG9aTWvyEnVC037O5mYvUvZO8jS2fQKN811kVR5Rcb1xN4oX5sNBHa06gM7TBAFpniZRz9TrzLcguslA35Q0PrvpTe7RaTZtpcusW28zBBF9jEoFefutMKWh9Nem+rtqSGDZhBG2aYcp5TL3htOTHllpHVybxuc1dXV0BX0ShehMh8tUhugf7zimreqMLVI6wbrfLvXxjlMb6jcaeVMzk2PfP2nZ25p6NQVXx5sdVU16kkIVY/JrBUoHzjZjNUHgGm/a+pdsSGOeGoOiRa/LMP1DMkxzdQceuVv05JoT/Vb3XQzNcXcsXMpfu/w7PxXv9acSuC8fGTyoiIE94yw18P3/uGk1ZVuvWlqjZUeirnYrG5hGW+193wFgcEjJFOBcY9puHE0mbq+yuyP8JXjwnh7qGpQAY/R5suKavassNRLMdwhYoNvpdduiw/bX1L5dRhvSFWxsuqaFpIhIj8hEkThST5JpUgFioCuu75TKTGLXgWaklI5EaTNH3bwc/Tq6fDm6vBG6qk1yab/uSkgj0AdeCxfWoqKwt1TK+dBqLel7YRnX81tfvprODvHuVW9kPqx14AEJ6R32os/BYlzhla6uVDN4KEvtL1J7/IFGKEoT9l2qrKb0fxgQ1FymP4ygqz8Rz7jH9JBM+20QcTTjb9osDOoep8YIDeiR+LalB4gCpEWIoDTVHsHwu6+9Qz5JDbp+51AcipUkTzP5eMMMu2FVt+mzX1143Li3oaoFH2WyUyHeKb7I8D2JP51poK0gzq4lNOO0U2SaOo1oZNMdecO0tnaYh3ndS8OhTfjeQQTpvdMH55vem/q90yppHBn2R53L+nuOjbbsNci641Hrkrp2bKJ2emv7G518ODsDNQzsr30DPf3Q4r3eoQdcFyZp7b5YZcAthkND1tbtHvY/bKPhkH50bWvhkBtPN40jZJTyw3DqVsMhfnXbjpF+cHwj/vEJYI5DRA1Tnsbjd8O9ZhffdtUH4WqKg0dr4iaYs0ePVnftZnqgz+PRPrm68Ff/CyLhpAfNRgAA", "baselines.py": "H4sIAFYSmWoC/9Uba3PbNvK7fgVGnbkhY0qRlLjXOlHmbDmXpk18TdO5uNFpNJAISzQpkiUo+dHrf7/dBUgCJGU7j344TxtJxGKx2PcuwG63e8KliIJYMCHzYMPzJJMsuWD5WrB4u1mIDH8to63MRSb7nc7LnchuKmCWiXybxZI54XzNc49FfCEi6fYZ+4XHK9FbAH6fwRpiSbgzsRTBTtAC74/fvuwseewHPs8Fy3CCxy6Ca5jBWZoFSRYwGGaxgFVZvo1hIIVvG5GvEx9AkyhKroJ4heg6yyTeiTgPkhhp/k1kiy2SmjMe9ZkzGoyeukdsuuHXzsgLH/UOXY/hjyH+Ojh0Z/1Ot9vtXGTJhs3nF1vYl5jPWbBJkwxwxHGSc0QuO/oR8Ce9YVyyOPXYFc9iIESq+TKMBDzoa8YVSH56K3gsPXYSSGAHgBcPfjg9eT85PrMnb4JrpKGY/IpvpQx4/FY9rsGKPAuWsoB1ZBCtk63IczGXyyQDti45iFmGAYgp4yCVWz3QYXf9+XwXCDlfJNvID2I1xdVLL4P0pi9TYAow2A9kzuNlSW2KD2BV/LDggYmyYqoPWpXEBUf9IM1BtTqdgpv9iyAC/hU/nW6wioGCrts5e83GbDhop/obUDNYKIOVLkBHBels2Nsgs5EREQti0sCF1n7Z6XR8ccHm4cY591jogcYKfzzwWDwP4iAfn712j4hTSt+1JJ14XpjGOCxh1YeH+uwnmznuWIwRnwvbyZ1zFxb7hvU+/w+o98U1k1dCpJIQKeLpAdEPKqXFjuvCv8A3vYGFQLksgNxFBCzsxWk/iC9wznQw89hZEguCQ8aFyKdQHpU6ElzAsxdjFokY9nFk6Q6YH6j0VpQPww3gtznqmpgQCay+jYPft8IJN33lO+auy56z0T3IJeDGXbFHaqe0SIXCXEeyF7RrG2OND5JorDCYwlZAWkUqyyo5Tapi60dNFg1rVGI5GLoKaWGcD0e5z5zrmJUBa/t9OPo2uy9w94ZfrME9JqJFcsUes0twoqzUYDDFDNyJNAnVZCbbHMT0x5/3KOfzVt2EydNw1lTHvl7QEjcAa1kTka1Mg0jxUwwPInGEQSTYbDesdIHk7tC9LNdJ5rPLJIh1jGIi9lP4mcs+xhqyEtTkaWhuyd7JjMA+ABQYC88yfuNM923DxDJTRnBtTsR9XEQJz/VYDIPONeuxa4g2sQOW95hiIkxI89S5hhA5FL3hSIHfEPgHAP/QCv7BBvf1ygvpwMwec4a4UAyzCi/d5Jgv5BJ4hOwi5ikWASLgGVA/xQ/azQrX9V1X71KLDhhSZ01hz0qiqG77BPp+u+IZ+xv7kW+ExIRh8MRlOXhxCVzdCL+H5EKYwgQD8XjsNyDMfzwqZYk7Pu/LNU/FdKjkRiDAplF/oMUNqUWEHgcg2MEDJI+rtmo+ITq6091qh82GoMpSVM4d/65QmM45yOS8j5HRGYBkHj1iI7cvtxtHzx2Pi8ngGy1rKWhThuWgMTuKbvDJvsLl9H5T8rlELf8jPCpn9PS3dKa2BNykbd0GqaO3Nj3qDWdesdHp8Gjm/mnpA6rAJTpucTO+lP2VyN2Ha8MX+q8VTxlGdthEsGTaf8FDW7kg1RuPSkcLSk0WoRKDPszg2yifw3OnCo6bLSmRlsgzdr7E38CvzZbG5x7+9+9coYIgwKNVX+5853wJtg2MmkNiDrmgkON/cpCbwvoRkSzZP2Bi/1d6EiUeWwfw+CPZ8gBsF74BRwduXVHvVdEoWX3w2EtMuWDnIWmrp/+/R2txqtIg3E2yIqHaUvPYYeXhyMM8GQzcKsTvYJtI5qx8gsstcDkqKZyTWjT4mAE4cB1TDzRtR/EC4/StGH9UBuwSs8BCNd/N1fo8TcFLOTWSP2YlzTBvAaS20d0gvPTO+LsaJXYq1tCapBHVsAyNMZn7DpodoJK/Zzl42gM27A/AJE/UjFfa/EqkvZLzddkoiYH3lQL9fWF+Pe3QEDzQxiqgBBIZpLeWyRrcDk0MARI1M8P1K1wf0kn4jJEk3FM8q2V+JSF2+pYJHprGruBKiy9+fh2zZzKNglwXs0alXOQti2A5l+lagN3xCJdXCwKbNDOAEiM4KA5DrWoF9fOpmoU+93JWeABi+GWlzGER8iSlwOCpwYlbM4GRiHt6ObP9eQORwgNywOgMNj1mg4r3mq26PugofUXDgXXB09OUEgGOtE43ZpMvGegw2KClmhlfYjgxNqTpNxUHYazlWmsEWPEAUPUAGgKwMg+0V/01DVzYBAw+UhG6AoDtuPW6GPIXAMVNE+yBmqgnxJclquKJ5m6KmwlhcOjCHB8AQjQDU3OBzB7AWRTErk5Yrqlo3Zey/CyiSKwgZ3mbQIJOOQuMKm1lnNQgSwKfXa2FKnijBBSUnbyesDQTF1C3siCvUlEIBFgDccwZMSiEUvdIQu2XrByDoMm+nrFCXEYF9YziNEJph4KSRiP4NdMyuloHkSgHsNGDgQXXrwRbTaNoZrn4PQpEOuLrvBd2DsWhqU1Q5tqgqhKFGVh3PmXkDl+Ma3TsVTIyxe0CjXsKOGzci3lKNlNzEADvIXG3IkskJU3wBFgNSQ2m0LbuhZuRZjpNGxVB8Ynbtg2zoB7dVVHv3cxivlzvIdlACYQ01qeZL9Smm4upicijqYGHEszZrOHfVZ4H5jtsUt3UpL2thioTNEp7ZVmrFsvyGI/SNZ8vsyAfD/vfffv9SBnEGFPnyu5+gBIhi27A8F5GIY91sXDEjnVDq3fKswiLmET3majhB5qVZskltv6S2LA6sjf6AAjT3B6WMaKVoyVDClTmjVWsnldG8mRghmam2jh6tmvXEa39LdXdKuJVMbHsew3be4n722C2ZnjGRiphQSiFFBu7Q5qSuYYyejyxuGrmf3c4h8JczaDZZktolGA436FYTFaBAydTE1fufkcBo0WaqOdhQH7WbnN/oY1/Eh074v2owWvwmRj62kaGNuvinVWYxJhe7xq7AKjnqlXwhQTfUg60gFx9BwE03jUGnVsg/FYnzxi6nVudLR+YvYriL89umhQdo3CKXrVz67Eubqvr9ssC0JoirpcizdlL+gBL34OwSIXKfWuDvLUpOh6hQRxjbkBp/VNK6zH9Gh3qr5Tk1VmM814Y7qyd0eI6R0ZrG2uK17XxYiPgoRJrtHrRXMa20znak9Pbbg1mdva6rK/pru52VXfEEj9I90eTMUjaY7t87eO3oRFQsI2bjp8OzK4ih/QPa6pEQnR5E4ScWlGgp9g7SakHxXaBuBIZ2yV4hvT1AkmZxFmpyblOTFQ+NzQyt2a2hk31ollM3y+VomPT+C9P3UaDL03XkK/YWwIKLNRjlJPqgmHbAGrLYClwjATogWakEV8Ks+FS/J0CPjoEgyIN0c88XN6GAbmCzoA0hepjeMz2DMiwLLlClp02t+jvPJai19WnZ339iWFCJlnuwFS3Uc7gBmEWbA31hPSUWhTfu0etUVxRh/nYM6K3MHi/6dwV6GPiH26ZgiSqP6X4OLm5BG6cGtNINTlrhGsnuzjPUYrWTm6rIgKTL02vpAee1wtJ2w/t1Uc13VLKKo4r7WoP5Z8axm16Hp5H1/Ln0rzr7bUnZYOk1i9WT2vOTz9Uji+Qic9zXvd78RzKPu3iZLDa8Hnl8t4YY6/pOTq3wv+d4Ons39gP+OEMv//20GVXQb5WxynlAT/UsPk6E3KdRKBRyplVRyotzRYiCMVIn9o7HgKh7DGkBE8huGp3qVUupBjBlHWADB1dDmM/UAPiTB12V1KvB4nFwDU7deXm1XGl/g7rD/p/h0VhIoG90eS9Kcgb9A+NYau80afLheiesQmGxTuT5FodQFx3j+7L+FFHJ249jk6+TsLvEdmflueHQmA7ZXpZDyGKUjqxQPs0u0boe0joVuMxTnLCdpedIX3TCWQy1Wo4ZfZ/xzZInyZ31keafXuLpHsqpBfUuHKUaR1gowv9vDWMD/5T3zWMlhYDNoWQlYkUKCZGtTVB36jNr+n2ycbs88FqhZZIklxcmKkOdrHg0ZRfzyhnQfur5uPjtrx3gi5WackBTgf5TKkDC99nD0udJ0UonVhZ80QlvpPywUZkKzT5raTeB6YJUuRGXxSlyu+RKsiNQBDJA7OlVZYqS+CzWSMtMU5ZOMreY63LFgGXDE8vr2522ZUiLgLMm0wXM0q7yB+2R3ggq2LcAgtERNrnvu/U0ivFOOPMhvwzzIc8yzXncZv7al49apMSDvc5jlCHjIoRn1SkWF7i88uRjR9BtGgrSFp6Wcd06J4FVKv2gOoVhFvqID8mDmDPail8uommulkyv4HsP4FU4ZG6U0WonLd8ncRYtPAUUgJIx0eHZS86UFcKIWQH/haidwYIqQ1G1bLwJRudPXbCg6GrnKNamYJ+lEhItFjKgwzRrHlU4CpQBBJysQgS5LPHYZ+xs4SVlwo72rPixcI4oRtDwUWwpEsHkdiJ6J7qCXT6kIbPWHHraU+pdNaskmph91uz/bZJdi1dbXU6pXhWPlzxIJaf1OD6/O5309Zo9cJ4nPK62KW6Kubua8zsrbHe0bUDHO6Rf227evA5vTGbzne6XVUdvwLBqmKzkm3X5rIqmPA+QcQ3C5+z/Ij1cuCZlbwrgaD7om8wjP2xF3jHA4LGGTaZwioQNqtkfYwPFKV8iepSoKkLEZwI7h2hOnta6QjQ3kEPdbVWaFrZLDcUTZmZaWK1q351Tzf5khPLZk08gXiJNgYoxQ1CAv+GYvidBQ6F6UItu40zDjY7p/uPRZyHxPwU3OapPra/yxAMS+Au+695yrhwG6bx9VT4HVd3bQx2UZyzHzwAz6KBZ1HHs3gAHpCtSnDITpx3GLrfAQfgB2pv2CyJDa2z1ga6G3Bzz3KPusYtmvwqbIEIdxABxRi10m05+RlrHDo1PNiryTqbp7H7y+Y7q18zrNJ9M7dW7a42m3mwrMdUOizqLoJlt/2KbeEyyxPVr3jHttGvxj5f7aa4A4Qskx3PAgx78/wmFeMu3sjolsnG6J5r4G3lCYpyNSfE46HoPW2UK3f3wZun9VhosFU/hcAeLC1MOx7R0AKZb7li5D2qYjdYRl17gZ8V65dRkDol1jkkMwtOPU3q/jdOW9VS+O+B9udOz/m5Opj/2dUGZZGB8M/vv2QMYJ5uoRQ3J+A5pC+oEUYXNarUdNG4urbvYvLaX0hIbObY+qtpKGhekXTO6Y6TkQFu1Fm/NW40SPSNvvYuCZ4T61cXnMYaG0shlHTX1i6Kfj6WMfCcSk1QfIi25cWSsHbPQ2986LX3qI2bwoCsv0zSG6fEVbQGekO3z2MYODLfGOB5zpdr8CWBhMQzUW/ACI4vEhTvv6j7FKJzRzD8hEiId5JLimYFKopnRcO4GsY4WeoqOFi1x8q7wu87XGtVKlR3mxfFOyggraj1fsm+NwX+Cj+m1KP2XkytVGp6oHuclnZuT0gR5y1+pdGIBTI+4e0D+1K/vvb1ua8eFD6hbtq/vHz1+v2vv+A94j9Uyfa+XLirNbgiRbGkO9FvB/R+4FmfoIr3BTTAKd3v752o+/0IYb0ooKFe0iV9R990dwGMbsTr0R/x6n55BIo46Fa0GnzFa2N4OVaPnauqsVvZn7rlVExtDK/MYTwFswGK0zcN8Pr9v06Pfz025usutR5/e/rGrG8BrqqXCxLevj04eT0xUOh6wLKTo5aEwGMqEzAQvZ68+TxEGNMKRNrNGogMf69hSvM5ABUhsVtG7nX+7PwPuegNaPI3AAA=", "datasets.py": "H4sIAFYSmWoC/91ae2/bOBL/35+C8GJxVCu7tvNomz0fLptkH0DRzV6KOx+MwEtLtM1aplySiuPN9T77zQwlS3JsN93uAosLEEemyHnPj8Nhms3mzVq7mXQqYhOxUImSlgkdMyNFwsZSR7OFMHMbskmaJOlK6Sm7kZFTqWbdU5ZOGCyGycvUuHaz2WxMTLpgo9Ekc5mRoxFTC3wFJHXqBC6zjXzo10SNi2edLZZrJizTS0/BzhMpjG7HwgkrnS3o8IWYy9E4SccgUpKKeKSMKh5BOhk22MEfmjgG7awbRUJH0uSLYzVVzgZ19kvQzKSRtBYVz2W4cWAgYeKbSCTSbMkroxRmWUUWyhdcX5w3Go1YTtjIyqWXnuuQzUMWhwyGQmb0FJ6QYL/b7gRnpAXYc86UTZ1Jl+Cf70UGcght2WqWWskiqZ1BdxnJlomIZMxsCv4QjpyyUFotsgVR8lORlTDCi2bZL/D1F2ZzbVgs75T3EDkSl12wPkrWLuaMdGoWIuEcJQ8CmvMVs2AlERPPQqQ0cyzTTiVVSdhSKLNSIHmskGIkUYq/9VEsIjVJDYOI0cwIPZX8uFMYAn8uQRa9bCdKi2TaRkH4xfAsZG9TLW9Zi10M6Slk4l7Zfi9gz3G6XEs+D9gz1pWvN6QWQOqyDVLxYDOmJjDsZTmrhRDGyrycFrL3XpBMG3Enk5HSsbznMCDMFEleBiEQtzOxlCX1O1hzMVRezve3m3F991iru/qyXfaPA5QXFv8V9Gq9ZjIBo96xFzC0WUvsnvdZp93tgf6gF3xWX78HaXa+tupXcCHJNcmSBJ2t2YsXbB58498Nz2Cg5Z/bNlvwgDh1afXAr7wDmaM5HxKf5z60gcPOaPJE39+WQbX1g4HxvgyMeXDrp609L0hSKZz3QT4h9NL5aUYCGGk2CNk6z0NEEW6XMgoRu8p0+4efyXFqyJzJ5AiiB9kLlkMRw2VqoiKfRx4rlwl+h3jGYNikDwAqmE4sZAihb0BWXJonzfXazVL9F8vGmUpcC3SbCTvjAaYEKBGnC0iUmC2lYTkCkRjWGUAimxPh1/9+98NPb384v/nh5uoKIw8AQDkWEdwCfLMMibgU/At/BSM4i7NIjcEbU6kloEFq2jk5ROR2ZKKjHkoh2ETdw6pJpj3e52A/Xrt8j5iqO3jCMQtKbgyU6pyevJNmDbaOZgDMtMKPmEx7lhANuQdJY0DPicgSN4JxXgrDJ80HtOLH/zyA/B+bbdiV0ljyIA8WDTTAvu2pdLypmyE7BeBoNPKsBiewfp81IcKbZWbn0OtXDpvz5m3on+L8yVPDRSF7BYi8WYmhAct2IjnhuJ8qkwrvFNROxGfxHxLv2y9im+mxSBBn40eczSHOtEVUeK+8k6YyXdgl7DMctqiQwQcgDmyfiAsr9qLPVh4MNusqQLIAVIYtgPMVgIAO2rD9rpeSK+0gaHsnFfNCSo7qis4RvjuFtq+R90bfYpei3DdizYeDaKhowRlXADxdAv/ObXshheYdn8xqB5bsx67fhlmHcOszsGvLofJDJpLRnTBf5lEb7XQpOfPpruj2/kBffNkudGgnwr/P/kRelUmqp7B7PM7SKAXMPOBWfF/FiE86rPPndhguH5b6PDJ0vTI731edxuS+b9jP3hxlffXB8PO6W+12HqBJCduQRm3qwLbFcil1zPNI4p8OIaysAvZ39nP73T6UGViQ9LeEjcaogRPBo6iZq8NRg+8/K2q6/09RAxb0BgITuqPm2aMsl9th5fhRyLb8+gIFth8MvKsWBxsnbVhkWoFAiwN88hm81W2/xMh7uc3sAPUFVmg7aONm8J59zY52cn1K4AIXgyw6dLjYDYWfaaUNze5ucnSKeZJFDpnEzuXqgLXl/RKOiXA0BZUprrdEbmHuH857+QekMhT7j/J4ciiJJ0ZElSTWmARQTXENuce7oMakYqUB6LnubJWPdn9FpbGkxqzTtlE2TkI2UzA+6NDRuUO2CumruMevz6tJqbei2y8Hc2tdD+qtXRpFHejb0pAAyHi00vDLh2t4WyAFEmp1Cyyp21Pa+na6wcNPAN8R0D8io/ROf1/QO6rVmWD7Nfw6Qq6Qfjv7AOxoC8BsNoZISra8iZgJkvdI+JNHstcDGUiAyBjO9RnrzQzU03pNbRJiZKF/XX26QyDuPSEZqj5c70wAcGjVXfVNbO/WVc37ad4d28r9wb4iYSsMn4Lag0eIHWIGHCb1aI8Z7N5fDhIRWtm0+fuXP4O9u4Eu6pdadXRKCuMLqmm2y95fpUnpYIrx0igwvRTbCOz+/VMkmbwyJjUc/J+f1H2vBch02XZc+I0BwS2RmlPrTX3IIJKGa+zYdW6LDSHv81RiF07NkyQVeMisjK6DorfTaDS+Yq0v+/ENkEw5iZQaN9dXFze+IkEP0dc8q4BXl61kkhSNWBk3MNnnmOzDHkHPSUiVV69ze7bpicbFe3xzUryhGOBFj2KCf0fzh/nHUfwQf4SRWEWOz/uI8X3fZu6/QlzQ/VedDprsK9ZjeWdiie1tXmkP25Vcggup5UTcCQ6P8eMIP+Dkl4uBIhT9DRQjfx7ZB1uR4gSl6HkpbE2GI1a2KYifIX4naAPUuMqn0tBAVvR1ZB7MDkakR98gq9cFq2OWH6AZHKAV0qkwPEJmNV7lWZuYwcMWr+PHvDZqnbDNsY6YRKVWFSbl0Q9Z0LdR9BDt4IHnkn5UY3HK4AjQKm4FGF08YGljfVAhdiBPRKCwhLSwqBrDvFaqiLM5UqA0xZfRA5LaYWIc7uNHRaotKiURdzSKu50tInTGQCpexopuLxnVRKTJhNTotAF5Ou0j/DipyYy1kxcYnkYPCBSTZxA4wQ4zYtnUn9Si4hXz5QJUqzMljTDRTEUi8ar4QiIsSopR3CtoErkKmdcMwYqaoIxxhBdGcBZseWOzT9V94p3kcX536m+lPUEjqQ0PuY9qqU9ZX/PRqZfVt8Dxgm9UXK/xagu8evFH+rQ9RL9bpWxq0mxp24yLgL05//bqzZury6LxC3m9mkkj/U1QIqyFmMy0w2YyDPmLqFTfYfWdwoYEcIlN4RDv/X40MMnO0hWCQ+qYSFZi7XvLuPtIZ1SE850wU+l895iPA3bz49vv31y1Lt6c39wwee/AvaUckA0sEWMAXPCuFwi4UEscb8CcZekq70S/m0nYY8RG+NSCv+EZcZHwvbglw6tOq1xq8KI0nUykwfa6IHMSpfmzfre03xl22TNsf8/UdNaK1UJqS9qHtezFBDdGJogFZE+/KeJdHog81yAo8pmJOwnEpWaxETBEd481NZfpMksIw9uk1RqVImJvf3oHck4zPI84KbGdD/G3SGORKLcmZ2xbK0JYgT1zXaoHpaOF4I7wepfxGVhlZZRDM1m3TsAmK4VWI6HoSjVAynQv6WXJL4sBFolcvhEm6+LWOcZwELDW+GtmiBOrprnBHL7EGIS5IIWbtYuwpb947bjpBWD66EXIEp9DvIkx1qzcGkMVwJv/At80K/fHwc4LZN78lq6Mmb8yLhbU7pGD2/plZRJzKMKwwqpfGPOgPVFuBKGqLZWRl3TLXRZmoEVRhnNUYIAXij7wgW+tBCrGA8rrDefyPjvvxg/izxNjHdMVqSfe2BaqeUm0wQoDyN41NRaLEuzAjU7eJ1HxfV65RrNUQVGJKg1i8MbLTofu4/Auu/+dgKov2Me7BXOJ/xDI3aIQ+QNJUlxMQ6wCLJ1BaDFwFIA7O275yF6adJzIBQWdYCf10U34ODGFxExsHkA560739BXw5h0qhqEkfhVQKBWvXXpcvvZV3XFQjY6FN5GycIxG6wGH3d4n/qDjwmuIf9BYOL+iZQFN1xfnrPYPC2iNSlkAlb50QmnL/nvy+utNDcQQKz3/axAMqHA9KlfRJu29OQJWTvY725EziPf7CcjRtv+J4LsOKoGUK0bFNeqRyNw/BcCf+X9Pob0W8WYfBPrS3ufFuEiMWtIW+TF+Qn6MKwlSU/XbixYAPoAU76JrMC7HQ5ju04c6XrdheT7i9VeB72r6QxNYINjNAU5lCINQ3Oxj0tnPpLOHyaYy5V2K05dBPUrXdPyMdgbnxLu4BeVqKZIP0o0Ui91sL1eFMxByCSZXT3DAquIA+h+OVdWOj62G2N6KuqV0q6dJpwrpcJcg6dQTpFPb0qmqAx5Lh9tRC/AptaKUUD1JwvygC+Qa/wOrtBkNwiUAAA==", "run_experiments.py": "H4sIAFYSmWoC/8VYa2/cNhb9Pr+C0GIBypblGaNdoMZqATdxirRuEzheIIXXEOgR5WEkUSpJ2Z4a89/3XOoxM/Z4sm0/bBDYFO+DvOc+6dzUFUvTvHWtkWnKVNXUxjGhde2EU7W2k36rthGzy+GHU5WM2Bdb64g9CKOVvsOuctK4ui5HId1WzZIJy3QTsUboDEv8b7JJTgdXbelUY+q5tBYahtM/QsVk0BrnqoTa4ZMH6k7XRgbhBBeJG+EWsdIW5/JphEt2O5kyWlSSwzJVwq4w7A7MhBNWOjuc9Onj+ZtPEatEAWuMFGU6cHT8t1iXSstR4PL8h/efri5/jVhaVB3PPCsG6pu3P3V7tiglLhxX0hk1H6W5yL601sksNcAitXMYErFxs4IXcAWl87qnTdi+f8ChEqX6fZdoOJlcnn/8xBKmtOPARep7ZWod30nHAyIFETuZAhhap5fnZxd7eD0dAjMSgJXp5SvMnkaM330HzskkkzkrYOyd5M60Mi3CU29TWUNBJR75CWLJE9gR+zb0tIXqabM18XAgGolI1axU1vFOb1lHJHLIZuOJHgG+jFgpbvsDK+hcsn8lbLoby78BTWUla2qYZZl8nJdtJjPm3SkfEYFalJ1i6xWqnFWxbSsesn+yE1YbVkrNdRO3Wv3W4lri9rq6CYk6Ox392F8/U3PHzy7fJ+DXAjl09vN6/cu43jJ5lNkRRHyJs7y5dObLsCH1r4fZ16T9hfbE2rb84ATT6rTWkn+pBx84s3yBRLrJ5omAXjaOnftfqEBUMOTpbp9BEDmaMdvIOaqJRcWpHStUWTK3kMw+SNmMJ/ZJ6IyYy1sxL0YCFRLrMmlM/GBQw3ge/Pjhe/bu7P3F+Vv2hJtdT2+uT09uyOkKlco6oef+yiBEnIIxcm1TSrhbloiijrJiwesJnAdGNolXPrtZnbInufqPDhDG4/3inDB3KQBBkB2yAPRw16XzsrULHj6H9vqm90T60hUEGFU8lGUn7pAbIA1h7TcSFtilDtb++hwxJJQrfG4Wko8a1udSyY1YLijXiAzDom4xvelcC2xOn7Fv6CXWkdqpCagodygWFt9DLSn61KgfaJcs7QMC4JWIluKokkKj2TgfCFRFWMH4bwhc5Za4di6NhA8RQeWy0+Wm0ER9LaYfPZyIaeyi1vPPka9JrkBxCzvDY1BhUupBK6i4zDr3I3t/l6a2nErCZ3CjpqwvHIumkTrjPqH7fpOM2KlymeCXPyHxDtLJoCZLPsd2IRpJyO4Ora5eJnRNNJ9FnSXBh8uzNxfnAyS8OAhRoIt0IRyx7VaDiCEUkg08UKAdWuzBwVZ1pXTvoCec0cub2orSb90LowQVUzjIdwbGO7rM6AZPq/BmCLnnbeTs+4uzq/MgXAfLqOxw1HbMfhAtBgehMWeUJVR6RGmdBHc9KQjDV0wclJy9ZZZmHevUfNBBG0kgsq9L67qfYko1p8gCNiYe1NCHnLvknUBUfF1VWimdzL4dDem//0fBb6bbgt+gV3cA59ScKFTh9gfE4gjmGt6dwf+iZvvIACOO5ZeJ7/WIU3ShukoJMtkF7MFB8RBibnMI2i3Zooj6jDIxlqZPoC0e9FojwNEDqPO0rBMT+xUShQou8x8Ltbn9akL4SpM6aZ0Ff7+i+FdaOex0iz3CVjapvFcZlQvwb35ipk0rmQFLEIbl2uTdnWwXItFo9jAE7CwhlDTrKv9XSsnr5m7VmB3lZAgkXz+KYo+mfTXEm7unmIyD9xjAGnbkmqJ3mMFjtOrK8vDPRPE6EnNNpb2wz3rZc9+9Jr7XX/9vXxFmf9FRz/2zMV+QUZgwUL7TlOzA85HGhjSthNJp2s8O1GCB0/MK/+GXi1/7aYbavm8S1jvako/9w4w6A6eBzusgErKfqPQdW5RcKIqCocxhhOl6Dfky6uaXcK2yO4Y+DX12gwQ9btZtaDhr7Wyv83BDqZ9HNrRuPRp5uEu/fzz1h7z56ePVLjT+fQVD4vgYrwt0E3tsxEPar+O5vQ/8/BfPi8Z1w1Bj6PWVB08UI3TJcOXvGjE6MXmin6uI+RKdPPlfK7QHPykmVwiUPTNP3bpupqKPB+UW/kHOT0L/dMdyDQ9Zq6JuDIPNEu99adAHOLHFqhJN2uLlgClVZryfQqP+ovNFqwuLF0WCh9t2euEGMb25kC2keruJUFB0zz32d7xgKeampy+Cu8nit/DKO0N/BoC+MHZ1CiQ5eYCyM5OPfV9+ISvLbVR8Mrw8oXcBY0/qcLY63nQFpvny+B/T03iWr1hFyJSiwdwTvfoiyAPphBfj0BYe8FHbkTqahZvKdvgxy3HjFyb3pOtA3NoUD4XgBlw8y2NfE2AVlv2rPAYLXwtYdafx0htkdomMvPJRzN0z1Umypdu6ZYNXVVmL8VKDO/54JuxwXueKwIp7mdEQlHdVETgFwP6Jb/jyyE23wQwn/wXypXgsBBMAAA==", "cdk_geom.py": "H4sIAFYSmWoC/9VYXW/byBV956+4dR5CZikhdos+2OsCja0GQVEvkA3aBQStMCJH0tgkhztDyha2+e89d2ZIUTKdpMUWaIUg5sfMnft57rk8OzuL3ktdysaojDJR5SoXjaS1KFWhpKW1NnRz+9dpFH3aSsrlWrRFw09OF+/pXqvKkhTZlgop1tRoUo2lalmtqJLCSNvwi520UwgTDZm2kBBjDB8kaG2kpFoYAW2koZg3JoRTaKOwiSpNtjFt1rRGFLRpsbJqpLyMBD1MwgH4qzbblW4NbYyot6QraqC4UyiTVWO0yi3rTitJtx9+vPnh7m5282l2m5KqosetgvaZsJKaRw0Ruq0t6XXQm3AGfJAV+JuTlaws7K913RaiURrmP6pmq9sGau8jmW8kjmkepXRalCR3MGwlVbWhBtrKfErEfg2iIIGU5aUVCWulwQpa7flBFJyc4RDnBF6LLVsIbLYwJ8NqtVZ+gyAj72XG2/mclLxhkC2fRNYU+4id8qDgWxjXVl6ZcKYKPkMYtjrHDmURRkTTyFLvpEsFSCp1zvEzsi5EJp3WXSawEwheslZlCFVt9JMqVbP3MUFSGV1GmS7rtnFGczh9Eu5T8s+hzWjoLqOIaCNWyJmCiGKRIkegDTzg3K3Wa06Ubj1l5NJYeVm6hmdXoiicipB0+OUq5N0cIheX/ATCs+TnC/oOlxnOweX37qm7dKXBR00hxyCi/mckp8JOUp+JWw0vOssvv0HfI6WwMCu05RhrWml4VVZ5jTrjgHDUYdTeZSXeu8rTnBGXR0JK8RR7W9JgSNLbMTAiesfyWVZfgxMuSaSoe2Nhll0jQLwl6AC3bmQluR5rbRXHMo36w//24yei76+JPt699xfvu8DxzS081VZijx02WIfsbkSI1azNCpVLGFmqSpVtSbYWVeVqh5ECicvLDkXNSAH1ObWNhJKS+uIOZcbFI39pZZXJDtjYxQdUuXQVuh/UY9RlTtsgKyHBAGZqXeWuIoRDNOCaruQAEF1kG4a4R2GpK64qj7qqhD7/4BpxVTi5u+sKx+2xKApZwizWbS0KK7vghPLhGPVVwrWEiCnzqIBaOWpVVFyOomCd4h/i8uffJ74iywBjaVRyAP74B2JPISvg/rYq1IN0CnVxgZuVqDYB2hjLZbH2GcegvUfeRFlrrJzo9SRX0Ni6WuZCP4KpTNumiw49SlQf+oiq2CcOKCmfRmdoRM6U5XLdIhZyuSRV1trwikp7nLBReFS1ZY28R0+o/S6bqXo/RYI0ShTTzgudhIwf+IVZ/tA9RfSjKIIqHZ4sOW42vkl8/UClGQeyUBxgn21dAofm4h8a/eiaxA3F9peWe8Ok14CDn0zZOleKdI0YVDjC3d5e4N4pF9+khH++D1+f2V9kl/1nfukMK+cLd8mZK7j00P82Mi6TQ7nzq9XhlQB0nUNqcgwI+gHCPplWHj3lvdm42B6OAMT0u2ucztF0lyt3eXsB1KRsgfNwmaW0WgBi/FNcTuhcTs4vngvstfkLp/no65WR4iE60UI/PJc1m4oa8J7HMR+aeLcZiVyqaBYCDZj+hiB/CcKfxfzF2B6F9v8hiNwnbn0cU7r1UeROcfu/FEXHKrh0wVpj/DmEkcHcwZw4aWAdt3hOcPdT36z8zfXrgAOv6Z/0Gqni/j5U1WuK8T8zHQPO03EdHMIY7RUKtCqZdtp4wZxzyyXaV7NcxgyfaXfWWTjrLKU3bx4eB3GybS1NnEz7fe794TWkTEO7uA7SBof57PZH/ZR2gM+JnzomPjgIETgSdk1nMPPsOCbB9b1SXv6o5H7fDRSr6il6hNjH85/mxWJaAsnit55uFJyffvsi+ZI2nYuONeIiOgXsfoEsRuRAxREZQywY7LfyxAGC++rfRdHKmTHaxAPph31j1T50n2YqH89SepD760KUq1yQvERpyflbVJucny/giyh6RZPf7gdp74yCiRPRbphQoON2DazL/1dY9AHTBpMrtPLnU52gdx8/3L6fBbrKg5oFByuk54CMgWuhCnTsS2ZBEOcoSxhAuP3DLaADFe584djW7NSOeZxHVMaeNauV4ViWz9wGoqomhTRwE1dtKGZPq/a1nHwgybHgyaIWG7cNbOyB+cRaGUZyk0sDlvVnb7gjjVtWzh+JiUchDR3DunByJ0FFuIhz1XZEsG6N0q0dTmgGFFKBBUJaz/94WrQgqQW6vuT1J06AWZnRNjgA4EQZ2CRyCcfmE6ctxAWu2dtaQIkiCeQ1xDDowKMXlM8Z5Zgf2wbwxhTz1Qn6XfLsh8UCGMy5EOa5lESee222nJywoZ9BmE06KgphTuuAeAfyy/7MWk/qFE/mDe1l0w010984jR2wee1tXKbuDHsA/k/C3PM8780DsWT1rkLpBY9Ll6idB3h/37dFfs9teb5w4LQcNs9Dq1Yp+YbEryXop+TJPx5qEmTNxaJvYauUVJJcucerxbCzqdDZuDb49Mn5gt5QeUWFfuT7t+GWPyVcs/oxxDT+zQHt87WNjdbNsHs0IuNWPHcvUpqAOijkQsw68KNFEszqjvdPmbzox/66wTlX7n/67prO+w0+l9whxzDZpkhbPorV5dds0tEKke9GKQK7d4egKnatakYJC14CyGs5zj14alRVK8d2Ogt3C94+OR/f3i9xHth9yfzhz1nZx9RZMHD1Dn5ORvd5Pzxjbi8zJf6xZi1rhlqP/U3aKZ6c8iouR5zy3Fqvcq3rOBnz1UhYu1/dDuPKSTi2ihWrj9SsWU+v77g3cGyw7U/eHuwYV8H5DiUI6a7s5rVcvLgQdTMFwMUxK+IKN3X81pPKpK9rM86Vu8Qxo4njyu6ImeK4MGGgEThWsQz9hPnACV6FnuQbqps7fDt61ohi/20GdogV91uGZSNLoSp7GCm/dfzlaXnpB+GRaQUGu28CA3YfTHPK9xPNcmykYXjyNvZ+HcIoI9nAtyvDcgZw3pGj5IgMchKvTr5kHddGSAaO8Mp8hVl1e14dyAV6e8EMoGJK776auu+bniqgqXkN+52zi2DpLMEk9KvPpM8DWi4rLPhVfL46YLBYDDsDv1aXFBqNGmbe56Ph7ymlPb+dncxajCtPfRvZh8ay7588Jd+A010dj8CAQ2I+mMWOlSHCsnOBwRo2eLxQ+Y0rvl1ydYySu8EYo3J5NCaoTuhz7wzit3Jfs1c5tt4hiikLUNX6yH3q5fmXX99/dTxm3eZq4dgNW+vu71+AJbbCoQzQ/z6gjLt8Ee9k58KZ/3Yy5+X8vWSVv4x8A8Nl2u0Z1gsvYC7GbvlS1cxcaHj1EYL1RXgyY1/E/u8BvroJgtHp4937pB8U+hkDVPwYy8D2m/30eCr+NwZVhzH/9QH0PwTSF5HxuXPHesMs+Qpy/QtTkx51ohsAAA==", "cdk_hull.py": "H4sIAFYSmWoC/41WTW/bRhC981dMlYNJR2JttwgKpQoQxEFTNEmBOA0MBAaxIkfSWuQuu7u0rB762/tmSUmW4wDVwZKX8/HmzZtZjkaj5I01d3w/WXV1TY4X7NiUTAvrKKyY3lz+QQ27JVNgH/Ik+cDKd44rqmBbBko9M7XOttbroK3xxQWXeeD7MZWq1nOn5JQ8bPGdTUkbqmYXEjxp2U24QmzHt/1zgjmT2AfXlQGJavaeKhUUaU9n+U8Xk7P8519IBTK20UbVODw7HycCmO/YbclIJWttKqQq667SZhlLaVQoVwDeGQ3jBlk4J/qMJzhBSrugsLGJqm5VySZQyXXt5VTJ+aRhZTy1yoVYp8BR1Np6u7SCwvESp2OgCqSDT1qH7LpV9UTdw3RuOxORzO39SzgemK6c2pgDKJQQ0cKOVsoDtfcJOCutM+x8fGbnnt1dXwqQlMpIVluWXbsd02alyxVqX9Tg8tgh8QHt8EGXAIDH+g59tdHkASAYOD3vpEoQ9Lv06/wsGvEi9lz7RCEiOEIbIrU7UXjVtDWT1/+wh1g+cau0m8YaH2X5puA3f3788vaa3v31/n1sxUPYsc6hWVGouq8r6OVKdAl2RMTku7a1LhCOdCNKUndK12oOSBsdVrYD4iQgGhqBTqqGA7sxKTMgl3OE5nuFKu+0ikkuwVRn1JbAijLLro6KnkYFey1epdCskSQM+BCS4iNgQrxyZcFWhCCjMldzXeuw7cfGSTRVJ+gDdEN3tu4a7jGJvrQRvo9VUm/RXq8rhkcf9VI7NL3mkJ5nyVy5rSgYR7RhociDu9dzUB3pLS4/vL5GmxtGEEwspZhl1dWBXmQ77SWHTiH7mtswpb+ltBOUY8H40tmNMNVijEzQqgaoCKXqse80DK3s1YES50yGlZvsdgNHS8cQy+copHKljPbNJE5r7FM/SGB3UEWrsDcEVeu418f0MBFInRzPFsbUAYqWDQSD+TYGQbuhRlhjiXgBsZ/wAfyDURnEJi4vsTXqPkJcMzvBCeqlSH6EjbpwtqGiWHSyv4qCIASxiVMatYOB7o9M17RbwpSbtvfyqHSb+1YJozvHnQCPTbh8YLJUTaNqs4tbVmuJWuC7d8KPYsn4MRhgrf/G9iJJDnqY0YskKazTy6JRay4OLM5ioPz4NEkSqIYK0UTRj71Lr7NpQviAhU+M6g0kLD1InVlmk1fDdihrK3oYH+8AGWEEGBMW+UfZzUKlBMNSrYDhOvcrdD4e6QWOXj0QM3wM/TrD6XM67zHIx/UgJFw8C257eIj5QNgduUgdn/B9CbHT2/glc/7dYFcC6iui5PstcENPf55RajzKeH6OP32eL/C++jodA+6Ypjc06f+bnsf/nvTe+2JHwNu0ObZwiq9a7sFlXmH8v2TZI99/qfqBToe90rOAy2ImB7nvmjTbMSp3CIJpv9BGB05hlgmxYg5qz77PhOy0PiL9KOZJPBV57Jt/cNbVPWxxlmMtgrS0ZpNeoe9yaczQ7HYm8bK9w2Ywr/Y7DiiR2qdXvSS+nt/s3bOBr0h29hgxHBmbE1Wf6Nuxvl1PXun1Ca7MMdgHsJveY7AW8IPMj7WfXo/jC8ZQFbiLrxuzGY1kHEaHYiUEtD17NCfSOq+cE9mNaVFbFbIDWL3Y+WHzSFeE6EPMRwjZPa7yqSGWPKNh3kY9SQvs7Lkq19O47/+HfywZhDwjh5uUh2Xcv1QdtkVY4elyFZ/KHmps1ckrgcUWYVx2C9x7ulx7ubu6NnlitYCub3ZNWeM9SNbWO/CYDuvrsG3WxjzHeydqavTuIooAnny5TWV3z07idZbFPbMXbFGI+IsCr7X1Ykynp+vNA+2uN7nnMNyW6ajPNgKzyD/Kvmcm2cQoauNghbsDUsjyfcqYLPkPWGdeh5QLAAA=", "cdk_robust.py": "H4sIAFYSmWoC/4VXXW/bNhR9168gsodJraMl3fqSNntYu25Btw5Is8FIUQi0dG0TlkiNpJK4yI/fuSQl2UXb+SG2yMv7ce7hucrJyUn2ymgvO6WlV0afWrManBcd2Q0JT/i5Nla8ev22zLI3UrWDJdGZhsSKlN4I2TSWnKNG5LXRa7KkaxK97MkuxHuqS/HP1dXpq+JCDLohi78K/rqsPgwq/Bb+ZL3bWAMz0RulvRNr1bZhayN7hPP3RDC9N2JDelCasrpFqmTdi2AF13Bl1uGBzVZWNRtk1pK8I8fLWrTG7Jxo1Y6E0SSQhqbaU5N10rlFOEoate+F88jNeVUL5eDbUiftTq5ailYRIGzJuqYeHhZCInVsZRx7TE2sB0eA7pp6qeyF8FZ14bzrpXU0lWpNJ3756+Z34VSDXMcqgH8pxBt0QEbTzJEXtyjJi6ba5Z8KABNMG+QqGfvg6pPwRig43p36rdAkLXdSk9psV2aw2b3yW6XhKN+JS3FzffVn9RY/nheIdsOBkWbH/TVopPSIf1NtZNdJARxMQjOrSbV5XH4iHm8fi7Ecdh+yaiVQciHX5NmsHNk7dCW2S7ksxMLCisCMWMxKOXSF9zuqt1Ir143oHvQFRQnZ961K4Gd0R3aPDEcaNlbec4umCPvoQXY0F5bCXt28n8NmHGwKtBDOpJ6nZE476est49NbpWvVtzS2LFCfy+oBeaj0Yq6Z3c7pTQFcKKW3phnqOU0QQWNTtlmvempB+IXQBgVpkKvh4D51CvyKXWA6akEPwKRWXsS7rHE/kZZF1WCkyBtay6H14qw8PysWyKc31sfkMncPLgsAs4WlR7F8Q2qSLTUvRIxxKc5wpgYJrEv3bsSX6ZrRg6x9u0e7fwn3b+TEeIGnm8E1p1tgqQUUd8SsTftQCAA4Qr8P1pF70IeEoVeybffTxQs6wd9H4pJtFXOrNh1Ye6ecwhXmOJ9xaSOVRs5/cYKEuhvp5QRwdsB6YQaG0W0J6pSanqRHi5UB8eMVdntggsN1zPF+q+pt5rZmaJMiCeyolY0KCB5b86DAK2r38MdE2zDouKeO+9qRdNDehjngM4jVAMTL7AQCHm58Va0HD4OqEqrjjgIVWAbvLktLeuh6QOmE7uMpB+7uS3SBoSwnDUnmNS9k08OOT1b4jmfxo9oQfiQDzIjfyDybNsd1ZK4rUIBsYEI1RlmE+1nNly6bdSjLMtA0YF9FjuS3i0hAMFZvLt9BvYuLTOADCK5De2LvJ0168k1F2p2+eweiN2pwJYPInjRCt6Tz2yI8qnWi/EvmPKRCi5fiR0hdzDNG548lIA81DQssqOBeHo3QLnEqzqPD19gKmHIxKYjuS55zQEVujJZt/nrBa0qv437D/rAAELximNhgxz7B+wflLs+LDxdp5WM8YU2PMyg7x7kjjdbFVFmw+vkypPfT10oh6mN0gObQy7zZIVo4w+djvPHIBzb/iM7VLejJdLgO+pMnYszt2mn99BnVYo1LCrZPvQlpno5qMr+ChAbF2kCKCmRSvqpyR+06keKSxWwhnjzZ3RdzMbv7EuMy6V1+EsOdLAQncFJMZm6AZOdFOTkObuZthClH7QvfB7nUhLas9ymX5QL8WlXQ+8BSPGEktYdkTQhcjdrOSsQvV6MjiDe/TrD2+kl+jicxZsvoKU2/Uc4+m61BFo340lScKJ/YwKxfFuLnWCznUtWyn1NOOAAAlFXWW6NqyuOhxfGZMFBaWdPlG9k6Ko5cLOFg+QH2H6fl78JbwZyd4vc+7+ei/n7HN+nPX18fVEWy3sbpvpXxVWR2F6y+h/6qT5TGB6CUaxZtfuULmNVRdZN8hxeCEegDVxPkR+CO0+argxyjph/CRPUHzqZA+MPvC5zfhXjksEDxEcjE37eVDU/5+WkgW/G4fCwnP0uPrUNZXC4OCDqjHUS3YSn6ivrmS59OumGVGqc3xSEpkg9Wv2MmpBt/Xp5Ny4H2CHcs6XOQP9K3l0O6GtH7UcDoBAzgC/O/IT3fM4QMfqubECt4GH1/VtGW34eRIZMAJU3rrJB8wYCVfMifl2cQyuDz3wXekZ49nz3g/w5Y8QQsO7mjamLA1AU9tO1sz/85YRxrzBkM86AR5XVxXNjSsqrgJh3lOu0ctdp+udeh39/utf1mr0MD7Re6Z7/eveb4uK/4/BnA4z7asYmCoAFzh+CvtXz4CwmE9jyNc4u9YTaFDhcvYsuwdX50gIcYb8Dw2fNwK3PQQzwNngrxw/jIRixtqc/H8PNnZUnuPp+A3/CV/Qf0dPz7Ng8AAA==", "uniforce.py": "H4sIAFYSmWoC/61YbW/byBH+rl8xTYEemdCKRF+aVjkVPTi5a2GnDe7SwoAgCEtyJa1E7TJLUrbv0P/eZ3b5JtlJDkX9wZb3ZXbmmWdmn9WzZ89G/9LqB2Ov3lHwb2EzsRdlRNciNxtpRWVK+gPdKDf4QVSVtJp+kqnZaFUpo2n6Op5Np/Hlq9cRxZP4j+F4NPpJqkORy4PUlcxobc2Bqq2kQhTSUiDsrTrO4stpPJ5OL+PLkNbGUl1KEiUJSqROtwdh96NElDJXWo6JPqjCfZyNRtMx/fMo7UWa1yW8oTtVbWmTm0TktL84SKHLFy+oMnT9DZV10iwrKcjkWtR5hfH5q0n4ZkREsHlQWlTyZGV5EHkOy9VWaHo/j19RYZSu4JzOyEpRlmqjOSJlmxm2hRM5SC2FlWUFg/aojkpvKAUMVo5H8ZgAM0mRbimFKZXxwYVQNqKKt9RaHUwmclU9kFk7axgyekaFNTuZVh7F7kSsSQyCH/pudONHpix2cIZ22MB+8Gh1Zxp/AAiHITO2lKmyEjqVbRDIUyHhYVrnwlKiSjYFE9sHnsmFlmFEANzvwYZScMLZVKl+ge3kgb1yo+3RsARCYT7rjmmAjkgURf5AfxO2UhuhvynhUOEwBSxRA3shRUU38+mUKnXAES7vB7EzlvE6GmCJBbk8ypxEXmzFfDKeTKbj0eWYvi9LeUhy2WLawVwWQjtwQEFOgTMqCPNqrcDda1uXexDL1poMWMdOefwuWswuSmOZ5pzIcgZnSWYb6VAvHd5YLDkxiPBuK3VTCsrZKkAmTJ67xXEz69kkggHnsgdScEFLZ47AMSr3qihkNh59O6Y9zUnXhwQAM3Pcicof5SNDVV5tZbqHo2Ij4Jonk1mvVaoQYFewwnEm2ACIOhmn5vByc2y6wsu2UYSzUZe+i559QAkQtwXio/r5+/cfbt79zF4NaZrI3NwRam/FfBmd1Y4A247DmmwoAG+KGtj0FFYVZQYftMEHawo2M0JdCfDznHRwQThkm2SRKjuSYugRJ3EkW3PMUtyZOrIxE0iv2FbJhARbneGSGXjCvdFbeVQO0vKsD0YE+rAr6LHbttlsVZZJPRtd0I/nDa1mngjv3rB9GJNTcNmPlFy8JGCHK9sDFZ4ccpJ3vUJzqObTycSZilwvTg0T5oI+DleetE8GZC8LFGdTMZlJ67bfw8NEpHtnyvkLRzgnpaxQIGpNa3nXOhOf0MK65tjSAuk9FAZk4splY5wO6j1KTSbhwFZxQ7Wi3PpKsrJJvhlcE1zjig/gSmafYQgXxBxJvQ/iiDS9fElB/Lw9OwwfAcDHERd7OcCfa921O99zmzbadO81yrfiHsCZMMZm7q7xveANVsChs7qr226wrpHmd3Wag7mAqbU8Hj3Dhe2YtFqt66q2crViI3ALjEUZeLKNmiH0hOKB71Vd+F1lqoqHMdpehZDGncPN8pQH2r1gNZN6NBoh9bTKc/OQBbcRXUWkOFt8i864ibncrLjfWKE3MnCzzRT/5CIB0M622x+OUZUAOpiG3Rot7664hxWYs+IhWNwu3LY57ZZjroFgEjJ1gnYURvRDEOIKh3C4WuyWnalHP+zerncvlzq4CsNlfzjs8sF5nuamlAH7wm7OTkwmIOe+G3HOYp0bsBKJ0AzM00F6BL1GWe1dRRcFr7ouIrJ6Ew3Kd37ZnItE/12n1pOj6wQzLm13D/jSdh3H3YhMwr5dtFdv1Di49hrAVesNZ5LEmhurKziYdGJuzORy2UB0DNOtx4iDvV0s4OkY2kOio5aBDpceckTtEzxgiN+GykTherjpOy7rgAPWA2CzGBuD4JYukMMnwVuG9Pw5xeG4rA9DwlTo+HPsd+MnqeSZ7+Y0+VL6Er5m5vQPADka0mTA4j4lZ0RQ2AgYAoYj3RqVgjERFXPE8pIPD8OT5VexAyg+QQh0O6L00n2wAH63C7Vchm1ZXU5ODXzqIYq5KOITSE7XInwXGjoLx0YI6ROg57HFZDl7VCMNDsGnqPOzN7iK+uTywjOuY6JhditdVnytBrcKEe0iOtQr5X7vGpIPbsbIKbkGWMttGMsQIe/xDLS+G4DDIt+MtbGHwHrXuFotgprKi2nch9Q49tHWsrNqkRHthBbYl3GgfAC9cMeFmIzHEzdbcFLhObugspD+Sn5XsXPju/Pxkl0GP5nchULq3Idd2HlYwsFvP+8ciwq2MXmqf94MCCewCM7BYnMUN7/Sd71CLQYc7FzBQw1qORepnP8gsC7sWyNnEiG11nYn1naPrO2+bA38KI5oTfP2qhg3fwMkLjU6RfHwhRcsBF4Ly0Fd+PBf+DpyNv4y9/QIhyTzyzAV3PD1HCNxXTcF5wAbPHVtdM5sj9xj7QmeRQxsZg4rJ7nmk77DNjvdA0Q1Vy8eW5VJIatEJormXvZSAQl/NYlOJMP7MIzaZodOupeyKE8VltdLnGLNoUxj/MfaskxFziqdFeyd5qeBlY3Coceq687UeYaT9/4N00mk9gXLTmbozJrf7FBOuAfwApWZjxSXhJ7/aQLfvTe5FEec/fqx0i5b3dvpeL5/oFWle5BRUqs8YyF6OfnzhXuKDmV6WVncJf7J514+Ne6pxnCIB81HJ3nc45Wcr6YuW+HdvvZxcC6cV+1jtE5yBXGXtWB019RtoxhKrxnAhHVuRBU+dYeB2361p8K4sbXCeDBkR1fB10XbRftqxNhnaTA6WcAXHS+YNgviZkHXUD+vBvxC/452HidKp6ZGpWBnxNYR1qbazhsh45Yz8fxqR6TAbwff3odo/G1MvINX8nU8aJ2/b55yutPus0Ypn0topzbWYIbT5HiCmbxm7g0Qih+Janre4fN1COIBBv8DDr8Riy/jcWoEUoRVvzcTLi7i2bLdC10Ds7FvoL12nSzDx4ij004fXQhTJwV+QSGULCJQTr3iulrwRu/v1yT0/187f1E0f82d66b2rvpq+u3xjzwl269e3GP7S9+5eFH71FcoztLbztOrTpn6fXPyZoIg4AvKfw8p+piv/UjSjwi+gcDStumf/ezlwzwXhyQTJGf0diHBhIjkYrpctuda7qYAh/1pT2lC5lsNMj0L7gf3v9fQft/ifkm/m9P9KVH7uXn7uRtavqH7fvh+eQ7/vT/5IPmbIwbk192sr5uOLgjjnA/X4X866cLg8Yz/Bqw/A+OWeeKC4m9k3IfkRKxbwUfY5DQmiIdK6VoOV56JzEXj9EIsl6yg238T/vcKo/w7WXrp+fm3odcK7jdrhZvwSXCtYHRt4m8SYyrGauHC2YWPoVk2IuYguIX8amd4M/AiyGHL6ySe5e6eDhoClhJsYLNh2OBq6uqkpJ2xhVvDxc0PBp8T1R+s2/JssssV6PYBetgb/RdjNgdU7BcAAA==", "wbp_baseline.py": "H4sIAFYSmWoC/3VVXY/jNBR9z6+4KhKkqM12Bma1jAgPLFqBBqQVu9KOhFDkJrett44dbGe65YHfzrl2+rEjGGkax/b9OOeeezObzYoP44J+1GzpS3qrrN56tdMLmv2s2Svf7nSrDL02Y4jstd3SBx139NrZje7YtjwrSuUf9dP97d3NbbV6eXd7OycVSNF+ySHqXkXtLK1VYKMtL+jg1TCIo7hjUmPcOR++Inewhe4Hwz3bmE3KLSKN66p1/YuPY3c8jN++Wq1evPuluU7tktmCgm9fzKuieA/PPcNzR17ZzvU6cMjxtlvjelhH/SR3/JYp8F+jIKGDIFMU3Cb26hO5J/YFct4rXOp0iAqXApWR+0E8jJ4pqvFrAsjJfc/K0qC0PyAiTbbzBQHCMEYOBbyz4TZFH5ZPyozIKRKrdkegs/MO9PdkXce0PpIdkWIqgLaRtz4T4zYSrHizDMIUOG7FNuh4pAPr7S5yJ8Y5ISBcDgrABu/Waq2NjprDgsALTfVB8g9FBm9JdWpI6Skz7NQyDMhKqnVQZp8oufg9M7eYqCvOMVJS21GB/shCu9I2xGR/EQUq9da76Fpncti0i9w6146iA5b6IZzERGo73QHnfbGk99fSaYWs1oUY6J+b6o56bYVTW79cJZRb7w6BwoiiiQKVN0dIxSUO6SxM0oH8CHJhTKPVG+f7ST2wXQcl4hTq4bpHMLJNqwYaHCoT4KfseKNGE+nVap7CCk0ngjt6SP5ZdbTx8CiHZ68V0a9qzSYQgsKVHG5GY6hTEdlFUqK0HTrUrSOIzOXdL0VtQfIBO5EeEij+hGawEExoHaiGN889bJIEUY41woHAn3KyEtEYd0ghBwWG7pOk61V1s8gKwHJ1R9FFZRbwPjiLwmhlzBG+O27VUdRxlslFEhLQcDy3QVXMMGwS+qbZjNI+TUPoeeeB0Fo3lb+YtsIRSnDnV7TCcJTBYoeiaN79/ppqnFYi7eojilCeXjoN/D2f39U6yLNEUG0Qco5+nFXV7Px7WA+N58HJGhNkNi+KAsUk2T8VsHxcgMihXi1y3etXWIGp50QtTmjr2Qn/bAGinv3ZrdcdfIi71ikfuL7FWnabg+7irv5mtZrfJ0O9oQQXBGEKCC0J1/3Z62mnQouxjyUcicE8XUh8b7TtmjWwNA8nwq+2mjeIzFZGGjcJSiMN3KDYRfLxCKrtUKmgvFdHoWJjnIo5gMWhYVs+5lcPDaTbuXWqqSsa7JcgcH6CZOmHzOQFh+4+wRQXq3bndMulnchO1BvVcv1GmcDzs8VjgMHjHzD8M+2hhfj+2WlOMpyyDNkaDVL/H+jS1jKEkuTzUZ1+s+VDg2Zb0PCEskPsKLhaN+L+M0bLqyyyUPA/CSUFqqWdTlqZnv8R86KdwNzVmDWlEJS+BuxDecPfiZ7zx66eyb4fnEmddCW8rLf0e1Hc9PxMdpflFVqAk8Bpnbe/ENTTwPqPcRWfD79T1bO772u6uZTJM0aBJbQRVPM3exek8Ah4JeCwN5jbtmrzx/4k4offZAKma/seWeb3EqLJ90I9Vcs22upY36A1si4b+XRyLYqsNjqe1Ztzmaz2fZVhNsW/nT5pkSgJAAA="}, "data": {"raw_results.csv": "...", "raw_ablation.csv": "...", "knn2ec_full.csv": "...", "uniforce_full.csv": "...", "wbp_subset.csv": "H4sIAFYSmWoC/51Xy25TSRDd+ytYgnRzVe/uXgKzmCwYodmwtDyJAYvgMLaDNH8/p9pJwMT2zYyUxNdt59Tr1Knq68VusV3uho+Lr6ubf4bN8tuwHq6H3eZuOf8yfF3uPt9eD1/mnxe7YXO33q2+LofXf14Or99dDn+8u5yt725u5p8Wd9vtarGeX8uQBwMPQTQIXj+8ef/i5e9vL65u1x9X18v11fIVTpnaGNEqcXMyt4KjkR5+T4DSFKjFGGbcanhQFfZzoEw/o+LdSVi3sVRv3CJCGkl9DixPwnIZqxYvQBYr7u0p7N169fF28/X5aVUaXaiZVQW0+zTkZFKLjNyIvJmoGVJwBvP5OYWjUQLgreCP23NQp1NqNrpKbS24stUjhdrpf6CTjE3cq+J7rZUmZ+Ema+N1bA76NNS7WY3jaM9MofRgQ62xJ2gN8YEAdv/7FHA6e8XHZu6hygB2OdKTi/Vqe/vMDOJcZRQCojdEzFLtqYuHiOeTiJRRGaNaK1VAxVrPID4zkYpGrCOq4kwkjOYpMok5kcsYwkcIkRZTKBzToZfb5bf5F8mY8QT/ag9ZTiVR6uilqQS7QDAPe/oQi6ewCo9WGjqPoI9iR7D0V7/0VNpQ3IqSelF3a03KGTCeAjMdTTyqUJRamJ9ieU//D8/wxo+i+VAbVLAo9KqaCZV6Do3Po8VQ0LeQKSchoxqKGjYgs4AzhbTl8MKRV9MKXcNbTa3oR/ivkl9FoqjuLTPNr/1n096tH+1JfOgjGaplpQijyY+E8hMgTQAi0d5GYnO4XRFQ01QNVfKK1pempAgjQ0zfK4QZo0kw+fJr0BlMALhCjSt+7slHT1MpJ+IBRUAahFMYmkAV+Urg6vXgJ3MXDIJJCehHI2v9yHJVgPfUXKw9MU8T5g2f+aglq9GCjGsBrGiISkNOao83Aw3WqNhO4CYq3NMRNSCzCndQirDZ7ffl5mbxbb411OL+zWPP+En7MTIEQfE10eadTFEcKwbUTCSaaoZa1NjYFS2BxqfoR2gyhunG7MR83AE+64D0FaloRZ0twFJKYyqQZoQqqgEOaIpV7iOYK+A3/LL9CVLBhqSkI1pmd+u/FjfzDXLdnxawcJ0JaPWkfdBPYhQkFQkvSDf1UoN8pCVHGWjV+QDHiFD+EOxzLfoRqAImQPilVOT/qHmeNE8jOgCWlQo5JLU3c2AHQTq4YB9ryQkQEzMaEkIccHZ/AipSRe7BEqmzu++LzXyjsL78+w6e4O1j+e1U+bHyaQkM2KSaiPfwC5lwUXDOoX22J7/lEBa8oOt1f9T7AOMeCB5H7fNZ+z40w1Tq/24E46V3FbiETacq9ltx7gkRStsEl1oBLfdHXjJzgc2yus2WN7frT/MrH/rDYrev/Rnr++0sKMCpkiovcTAUjwCeDycnEIZGgdsArFHVsnSObU0b7Vf1qEmnAGvA+aqM0de6NAcIBq6TocUoNGbrNIodHivT8PA80c4+BNZNKvDAQWm3X+8bxyB5AhLDBlLQwA1cX9AQvUQtxQdXJhAXdrwfOYqFoKXkq94fgZ5QScOpw/xqu5wrNpZ8QCztPDdhGQu5Nlgx9Vbsl2AO0PgsWs7zMpYG9hIkz5HhdBpjHazjnJxSvXc1eh5Sg4EAtuF+sT/KrSkwdgwErLP1cgsy9A2xPz1GEiebvI2WQxMdjdsFd/kuRKB0yf0OAzubDPcuccw2SGzNkZJHQZgJrmqauS2zy81qixvxIrdIdD7C1ZODmkdBBJAn9FXBih99sEJUoWLVoGa4QaVdLEuQAIwAzjtqFze4IZFyp7n9mDyNmadidvQQxhZSnftG6zL6P4N+g4C3uxdXKambh+g92qB0YrFsQ8UQMowlmA4M2R5VxpOiml5lj6XYKWQWi4dSzVb0PoBTfmteD1IWKibA7MNqvXxMe25veiLvuBAwLufIcu57oMt+VrphT0DC+jULQysbHs2PwQXpgji0sj/CGgejUAHPxpm9eXvx13K9+rR+yVc321cPHqiXDJ1PdCy2+xFBovSloCn7tvBD1n5bfVrtthfv374GRx5DwpDh01sfqhlQgRzzaMSGPaEeYCYrL7bL3e12cehpZyifaPDA5Ec6wI+C5uPDK8m/bg+j0wUSAAA=", "candidate_family_results.csv": "H4sIAFYSmWoC/5VcbY7cOA79n7MYBevbusCeYP8Xarprso10KkFXZ4C9/Uq2JD/KpNyLCcY1E74SLZEURT7V6+3z9rx/Tn/ffr69/3f6uP/OH6fPjz/364/px/U/t8/077fH2+f083p//X5/Ti+3x+v15dfjcX/5vL9Oj+vn/fn5nJ73l+e3x5/39+v325/n8+32uL7qKf+PaZ5+PB6Tyv/Yyfjp3+nb81NffBQh329/fbzd3ytMF1h6qksQUR+P70leZ0T6syLSc754I0DUrpuZjCq6pa+42ChCiG5m0qboZpJu1omwppz5onKaKFdGSU9zUSKi160MlJ7qssgw1K0MlJ5JNwFhxDUNMmTXTR/WdFlE2KbbOmnKFd1c0i1KEEuUW4pyS5q3WYaMDE5cVCsq520PUTPnDvuq6osOMqZTbym49FTzxUQZuCm4vRMaql4kDPiEm0wsCsY8fV7GEAXdtBRcesaLsjKu6ecmrYt+Oq+ukzCa6LfbnrpEJWM6/WIZKz3TBAYvA1FB9CYtDgbO4dH+9EVFGUMUTG8zVz+c1zX2MrJp6KmGSrQnSyJLmYr01BcjQ7rQ4svMp6e/OBnWtOvCni0G+Ofx9vevj5/cdpGWt/hUeurmhkdEt7jEeb2IwpWdC2LOpud5hJI9VwKMokqN+0cUBpVQpixkxYRxwCPCZMsulp5po/ASgmgW6OtYcZw2ZYGa2iwspiGaFUOz2dCskhC9ZvuurEQTMIJmybUjj7DiYrpFQoxWcwkSCiIwXc1+/ZkdwtPNVQURcggfdTdKH5QaDYbho4C049yA2SAsXZ7DZAv7A93AkvmIMNy+dh/dt68DRMvrGkTIcHcdDCVpN0vzQPMmDLzOipBeuzLj6Rkv1ogwSTmlBYglzlog6Zmc1YuQzltVy4LSB2Vli7XEY8smmZ6qbQyf5rAn6Cmi1+mW+RBZzDMj5rOqZSJEflMk5QRTSVoixA2QBLufyuzk7WJmBMmsnAm3iRgL7pYd0VHT7mMZ2V0DQydhbmkSka97TpzKWixJ0pmDJFjwjOumm9ETYVQjAehiaAaw6WGzMKzGcnzF3VoDxlV2OizRI1tpNVLYU4h8VSNMNc3JEwfffAzWq84l5uYTgOaE0TAsfrdmxatppGBRYkUSVeooutumJqKWE0UlUv5blM5GFzjxqgQV9UdRYqF42IiGE0Y1Ik31HPvt9ZQaz2fanPqqEF9Ppc+89RhNiWGolpNRYdEwvOPE61QQw6iR8/Z4e/7iEmrfnec0L9/lEvTc6HkMZBGqaK+y+oGVp2dMPKJXT+/l+xR/1yl5vOMxraqRtqIyhlr2AhIVh8QhuVBZMJN3Py/IE5U0Pd0IkDZLOv/ZxPNp17JvQLOF8vUm59sx8vJ9horHmlr+6TGYn+I2YdgxLFm5EndNgAymlydbQFq5EkR0jr2z4zHtBORoWYWKMwlzVxHwgnxnTbbMbXoerIlJlJNSxY1U3pz8wgKUOFNKAnRqufLq6Wkv2gggSa2gWIAWqxNeAnTxANRyl9kIIL4qsR97OgApmsAZNiWfgnynlS1apae5GAmEWhFD5N/dCvVhc7FWAHRlEkjyTTskHkB8bVjlXOZ5/339obNvpE/F1HUWXbK5/uv2/lzt1l6COchWXYo8OQ4FfRDPWhRRWhDpJFVTIs2d00WJ9MFmh+5lUYkkDztQyt+P4lWJJEo3nl4LXbQw2dxtqFORji75VNTL7lqs8rpokZ5riagX37RYPQrKkirXaFdRt67eviJuk7Ul+KRn8tijaNWiiu82uyuN8lmNIkvqLels3smqXY2A8cxmhzvIEj2SfAkD6ely5+Mg3/QIGGNU9oJOVjc9kumkaSir4nO64Y7CqEgCQFHR5VTjIF8VSbLgv3PeEFdZNV9fHS5MTrnmNcYXY7K5xZHLy0f55roVE8s8pqeZ26QTzOq5VR7OITZn9L24QpXSXBcnS890VA6Oke9UStnLXOc/fTAqJydHFCi1oOG4HON7cY1KkdirUtKgGflOKT8txY7TU4dLsAwGVPJoQy7XVzbvmzufymu8/nFl6dLTX5bIiDfvrhAVatoQcrKYq/VH1OrkFQH5lcv5WC+uqk7Z8CPqFNtSE3HQaYMoW80jfdBr+n1EFZ02BBzzbd7TenHdzdO+MS/5MHAUZ+apHk3Sh6TT4hkUnad9F7TZ+n79c/94v/2+PlPwn8p/YFBc0PNsPjcKCBKTFnoeTKcCAdUi09IVCBWPUCRcY86wiIguapNcLkigTbHV1sk+lrY8FtEHzrqPJMUkwK7XmuoTvRZxmE2xFUHqbinf//P46/Z+/Uh/s366PV7ur7iW3fkxrYqAIDNGEsC053sJ1aaMnCFVPlmwiLqWtsuwbU7nBATOGT1HrmVbAVXnjMb4Ods/i9BNs+yWqmZI2WWikiD9cu4HGZM3ZQGFOyJhIvhv9/dfj+/XFzetH26fbTFtfpVk/mUx0zPFVcPLV61sSZz2bCh3C1lIVqmIk3O3Y+VVUSkX45apxsj09PlwwQJ2ndSWre41YZNTNBa0abWNAqFi7coyAF3U8odELZ33WPldq83rsVCdgguL2ZTyJd7viVVuQmWpP8/n9dNM9TM644zbg8m7roAgzjjjAUOv5x4e1QLrTMtXWtCsOqPvzpV2LRnxiF2zFUVmzEogjBI4ZSHwCE2mrJiXzlFCKQnRT9leg9hYBjyqaQa16xRXs/zb83416biXPzQn9AeNPCO7m1U4Lt9ylN9MKhwXLvayqulB9iiz1v17UbTuiNWrYkSdeDXsiFPhdC9Yvcx1HKc1UTjI7jqs8iS4LMtRflPCdbSmrT93fyYPXEtO66eyKH5a650pgyohOzt8blQcxasyBULylTwjR0RWp0jDqm99/V5aEXXqQT99MBdjOPFeHUwGrOIQoA7UO7fY00vrXZ00ofVY5/IxN9cuj+JEHXusfx0RTR1L1dHujCWo7y98T35EFEQQJo5OnxH/GvL/4v4BClII1YrwMpkPx8MiQIgjPt+G0p2frCe6E4Iezsxupkqmaxp5PudW85I5d9LyjWl37f0sreRpe8qigwGh1R/XSD3g0aGeuO5Kn9LiGpQw45bWTRGYcQBja6gDohtAgeu2NBaEQHXjR0zv6E6Zaw3ak9c4yyEVWUDSQdUpGw18A2pHYUAFtENnNCcUM5wftNUYBiwzBBEmhj/hjUmeEfWAOgaOQfIjpU+4YG20jg52oKZozmA6RphxJ/wuHA05RIsdULyk0WZ1wtniZ1KmodnO4UmqocwZDYv4QsfEiiMmFroCIWOZM2aVFNS8HpGrcF5IP8SfkaX48dTc2qU8X0oaUMcz/hMOuFOglGosFZ4DJQ0Y7RmnCYyN0ppMi0w8rQmtFCssi5PZStVrO8KS0TxhaRO3vbiLMhmpafUFPtIXZTtjWJlGeFSyhmcagTh2u5yWGUQtqnWnMWt5EhHKYy2JYSnZ/i06gpCJPEFIELdhQPypE0u5P0Hg/uzStbIrMIUOi/w1Ws+XpQ9e3/F1jBb4Ors8vVdiB0ScM+Pj3fuUjjNaAW8Ekg27AjYOeTMYw3HXdlqkzkgZkAlDNgyfjey7GkOIaX7RMTGCHpJc2ki62wploosE0UPqipRkGy2yVxBCerVqSEjBmcA590bkpCAER+qoMnI60FFNYpSpJri2pA9uxuQRwBH+SFQyfwTHIpdwzJgRAmZLxlIDUghv6vNavxzxPAAH1VjdL9cXDhbzWiockTfgeEBI2tHK/A08UhBzDxItIyM4ZgaDKN9fyRn7qVznTE1gXFRE8m/C5naeI12ANM3HtcSl2BBrmZ9cnZwXjk4B0pQ2bEWaRIZwTImwsEwJECfRUMkEiAbpOBCG50CAOK38LCKzoUIoucEzLAuNI+TGBNk7jMxXWC2PpywYy1MWCAT2AZdNXGQhAKojIui1eMISEQgILNe3ZeS4BYCi9AKTc1yWXkAgEMZc3kVFysBqljxrQDWWSM8aICAgDtiLczIToKA4MoC9mIUnAxAQsJZcMwmuwd+9Ffb4I8eF0Me32iOSzz46bto3xyB9e5NnXu7bA4j6qz5pxIOjQ13ZDIBtNN/RKpJnnXTXN9yxwR4H/XXAkANgso5xw7y9Wd8MC4OeOYKQFp1MatwEr2oeUoxRHxxBmANZe9LX5ifS5NKi3NpGEHKyU7o66lZnWGlyg8/ofGdc6lgDhNy1d2rYg95gW3SDXGGl80ltaMSQ9t/sh53lDee7tzK5QyI1lxGCJAwbTrrFzaj67mcYNIwRhGWJXIMadoCbc3Z2L7eO0aM9Lasuy0lbF9XcawMrG1Xu7PITovNZU+jYbnPPNG0N27RFcawnOCc2Y+vqkn5sWaNjP7YOEGlfOJ/FhUbrhjj2WiPbakVpLLn4OGifZtCxg6oyzY/voCJg99Ot/S92RRGEQVwboTGKAHLD1QyanQ1E+p0qMzv5ficC8HqH2INcm9Rw3iUcIT38lRM802C27uLo50cQhVFRbMoaWUPdznj8T3YgCu8lOD/8LQ2E4TFqtsNfuIAzL6Gkq/EPTyAMb5qEZfhzEHAShXdTzI+YkKsH0kHUxsGPJ6CKmIwpM/hdA2kWl8FPDkCdm/zqgA2DXwNA0P6DALv1shf1ee1MTnEGd+hx0rnfqOFvtuNY6Chaja6cS/Pn/OgqOKLI+obRHW2cQjzYB/7i9N5aIDy5OfA3nL9c+wfZ7kZyWPh7w+2ru6vDLvDXe6t8d8NXReEe7l6tJRdE4yJcmN3r33CL0lnhZiu8LKbZsxWun369NC2rzl8MPFg3snKlmnEXGcilSy/eiIQqLknAXRRvLKJR77vavsrMhUJUjJwNjHzhT4rE0cnX8fhxdGuoc3fl+NCtG/2Gu8iGxUq8y2asfM1M2qC95u6AQTkQSsqGK+6B9HZfayvU6o3BzVyqglId0HFtjkrc5Scs0+H9J6vZS0pYc8N7Skqxd4mgIkZY8cbwN35IaQsv/WjuMovqIP2lHL/w92VIsQmvzGiuLrOvGHOlZeEKMgoQhwsnvt3p6u+C0DH2krzP1irf1IBKDMyYyc1S+RIFX53VudEgX3CQVjMsg8sHUOCASGdzQJHvBUCdgoSUZAQyZR/KDa4sqstVUR1FMj3UDcB0XL7mIJHdy/m/57s7kVneMAwZ3QyI4nBmJVxx4wccbjhWk01COsTrfqS9omHWXy9gyM/tbTr+82xZjvJ+ziU0ZWtYOjGcQ+mZJQisXzjs0TaWFXi5CCghbGXmRiswZ+Gg54rZr9xZFb79Dwbq7z+EUwAA", "hull_sweep.csv": "H4sIAFYSmWoC/4VXUW7jOAz9z1mMQKQoUTqNkW08s0HTtGiSAfb2S1J2bMl2xzaE1NUjqcdHSj6fHqf78Oh+nT4u1/+67+Gre3w/h/69e+//PT26W/8Y7o97dx+++uHP5Tzc3obuq/8YzpfTTd6+3Q+35/Xa/z4973d51Z+x0xed60BuHzt39C4sL33lIoeOj2EHDAUMCk7scmQmyD5QIgXnLhzDHhYL1stEwvoO8g6oi8eIO2D/c9TMwS0u0CXsxUHFVFJTMRGCCxg9BE5Zw0DnWlORWlPgKjK9MVffylHgmNG71yj0pLhna+RWg3DKUYTg0zxGoQf3sDjT45DrS19WlmSUQHzYM+ZnghbcECtb8q5eEnpd1C5BI9k4LoorpGPBwkjI83b59fn90epUpUGBgTC9RqObJTGet7Ewp8Xn+rZMKZWwDS1MkmYPM8Y0DyXJ4pXCNrTwRrpW30RsAmfReTxmv42mn2P2XTpiu9xGhpb+0Fw7UlotopEhWD3P9S3lvqloOvKeJfxLcaw1GeOOKT/LCBLbmmoRgiyJ/Q66UEvOqkPBvmVEsgojuQ/fSBA3C0hMObZ8phWuUGjuOh1WEwozFgqV5IodXE0rq0brVFjpiUxQWnoxr2C6XPHAWqvELqSEPmAmgGRhB+3SC2+1ikDjFrjEDsec19NGgWhQzTWCYI0ZF+ytDHwG78gTJ4wuKbu5TkcUVRGtrfhdXutMQ5kC1nLGFZxul/tn21yy7WO1GDQhoBXCW0BbRumNpSHO24dVn0gpHQNtQcdysN7eEGfZFGnWG0+cmKztjLVAFvuUWbZMawiYQBKdXqMke6r12s7YcNjaFVSPvnPJqgIr6EbDgebSYpH8JdoEwuwUUl4+aEA5AsAmEH/elERN2aHL0yihu7xpyM8NHnIAnAfcaiu8FxDNffLVIUvDxE1VSUDhoKe1d9QUyC9hEeUmjZ9cXt5mNaCKiWODAkMF9YLNXmyxdFly12AU4TuyXEMGStlHYIdgzQVkl8i6IxoozOFJpyht0/qNA7aAmmlg034siSwuGhQW42UjiCHNQwGJozQHJKQvIopWQlP/yCEGBJOiFBI4bR0NroRoRAdEXt5jRwTd7VoYzu4ab+w3ciwhx9E1uP4cppjBHlJISp5Fn96hxxASjt4xyR6wQsKIDJqBccuRufkY02ouTl6U0NQwOiGtExRhuIUGBZh1laVfGhtSv76dCTpTHkrNTG5n4mST65nMh88/w/f19NXfSf4x/jGlVQkKABGliGPmyEQTQabOTeyY2nLMqS4uO6tmNmxj8S9ykiMP5jg/kmCfDs/bP6dr/y1LtF8n+fA6j0uwDYUExnkxvopAjtybYF0DlaZIKS6fMJ10w45fNGiwDzI5X1J2kKPjRL6cOUOpiHQYrp+33/1b6OzH6WExi9OirjhpBLzEujUZ1E3pIIyenHy0BWsleUKiqHsLiQWp4USsr7n4RES3z8t96GV/tB8WnWjWHDpfPS9R6DdejdIwYzlobJwzCkpPWjWqhOiLAOu7NKNsR1P54B7Otn/aLwlRzgmdHbFD3b8LCuz00aKgoKzrx0pfmcv3W3UOcCTeCdd2sNix7pSXZ0MZ7UMAjdnMh/8BbCHhbEwQAAA=", "robust_sweep.csv": "H4sIAFYSmWoC/5Va3XLkKg6+32dxpdAvcLmXe7svkOqd6TObmkwylZ9TtW+/AtttwEBP7D4+mS6+D0lIQkb9/fJxeb9+LH9dfj09/295u/5ePt4+r48/lx+XX78uy8/H/14+ln/++1/L+/X34/Xvp+/Xl29X+8e393+8vD69Xx/BLfmPxS28uAe3iD0jgkQMGiVyVA32lTbXwg/UpwB7ghG5B6mvhJE+BmeY0zywibpjFGP58YbhPgbGGHlwfQzO5gktBjfZNJmRA2OypVBgcTGrh0ogPkT2pEF60+JNVKMIpIyeI3DAyGhfcUYygstcahTQp8A/pugYjG5GThQaKEawoWoQb18FVkhq7c+FzmtLlSInCq8Ryd2eNJYCv0BxMieXingnpObXPnrB5NdC4oEx7E/Tw/cZYMTQcwMu5b4/6SE2tfGYHSlKcKIYggAFn0LU11coHInaeKQ0HojAvMCLo4BE/RAIfQocUXSiVVsKqBQBQRGvHgFZkyIB1aVMo4LgKUJhTToHb7aFt3EeAokzXFJkg5JjYh/IFJE+Bf4xBT/ElgLLDMnAznmkoM6ZCskWvrpDke3oHNprRrEEUd06nrdIkVDfhvEthkpZ1cUYhDhFqkPIspbuB9ybdo/evG6iTD4EChIirRRRTHt3e5rRuU+xyu8Bk1WMRS2QfaaoCCj0lp5LRQwOFjpCXp15bKLg+u4tfRm+3pESWi5mS8CaFWkYhHpxwFUcgBcw01nqjHYlDhfLG4qcLN2AVvTJB41Fgs9xoIDIHKK3TdccM0Wj71PsmjCaESQ5M9h/Oa24kgULY8h5v+1TgEPHLoBZSYW6ilQBbX6F3tm8UYHTNxpOuS30GWBd1DODTYp9yJ5QVZSis7QVQ9AcfJU7Y+jJjcVG4F3M1lfnLRVx9sZA5cWLjihgSGHGNBj1YTiEMdS1VWfd9oAOOQ7Im7Esg6V4kuyTVF9GoX2KLaBRLYU5KwJs82CWXoU3lGIPA6tt0KoKWz3LnKsDnqTglqLcjolsZ/QOLRKUnMNMUVcWvYXcAxozUYq+GiTjeVfR+6DsqJ/Pz48/Lp/v70+Xl8fvuKQvLGggo2BP2/V6+bX4mUDhBjUHoeQgk8F4GwyRq2uGhUZGErZ0bd6mtsBGtOBQPWhkJESrN5QFLJ1ER3ehOIP6ARRbo2J9J2V1ii0kbh2P7kxbSOzqmw3LLTbVdT1HcFrf0DNVDT6Edr2p3RyNMzSO0a13OBsc54MLQdG2fyieMzlbj0gzwWhw6wMpFVSXGprn6ELOphwVA+scjH809efL01+vb7/G+eCIy/7QwlObjUoPd+tDsUgdddV2OHmLbFf7VDHe1q+PhGl00Aw6DyzoQ1s3SHs7H4+EjDNkkV7V3nGPx/ZCOEaWuVYAjwd21mWYBTp+F6ZYmLudn4Lv+KwOwKcUsO0M20aRUpdMoVDFNE7H1vHvBmPxbjqlO1iYYmGKrWXcbP5Bpyjfy4xTaXKb4AQqHLI4nkjPcGSVE6oI9fpcA4+1KUDnnO52a51GwR+NwjujTlt2OKLkNKyoz45MdRqFvSSaRo2CzQCl2YdxlRZsNAzvDTsVU219MUHVMuBoGN4b1hq7PJ/IZyZHEJ5htRBhNAx7wy4vT++v472uPveTtIFNgFAUzXUdwevR7ghYbCRi77O8P+OR0mvYadNbD6NGAwv3rA5DxevxJtADFh57YPJ5bpoResBT3DTuJEcq6QFhCsQxsLBh83LHR/23AUfxBs11lHNdYOF4oQ0abNZjGJSd/ZS7yHbNIcTygye7jkIVXKg+xc7TReIEOVLzFNCubgYUbx5dZFno1kA8UmcXWZj2wKzTZ2RqDv2UjLI/l/W0bTtzixo0QjpBDOpFshPuGWg9tdV8VNXlgCFHlcnS2VM68+ly4Fc4pOWAuS6C6MvbKKhPAV+g8H0K/AJFaClwrkizJn7tpnQ5hppQcxS/1t1dDvyKHG7lQFesLG51V/pfdCGw+aQDiRxczL4ad384w2ACA7cv4BmHc9xZTNjFhJhbXgROo5dIaEWzbmKGEQwmMJsORjicThdPMKyMGUw5UGG29x/vgDeYjmAwhQ1nwwksK/f69/Xt+fL78d3Wf9n+UWWWYMHro4/kvb1u5b4Yh0AeXSSHhLK1UCZE8AUinRHhF4gGqpWJJgRBb3kWAe3NCnsnoykqJjww4GFtTh7vyHM76Gcngdh0IM2dU26usUBYKebFxLErSoxr++SUhWm0aFipdmKiIvUkzlxYTohW3TRGT7YJAQl5Au7moLTJfb785/L8+GZum/+6vHy7fq+3uqAQMbC3GyRwPtCvq1azEs6IYETE9oJtK3k8c792wrR1OsDbukNMvRG0rXtQRg+Ydpdcm+cYKQikXo/6/JMOMZOVn3tEMCIyt6o+ml1gwoRDJsvIxWc9Au4SYa0biJilxaVVz+5NYHErHJ2ZzooezU3SCROMmEgwMoBX9hZCqdUrI+WwVu7EVIvEWaTr8+vLj8dvsuQ/Lh/ZAY4OiQCbcZxEW32D5988LGo7+xC3tvvZim8JqUnm0u94MPd3Fm/WHAJxCAw2o+sB904g5EQQRJyzILZ9QCmkr2xxnf2JMXlx7qjxmAZyb0uNgtOPPHw0w0vqVa1gx5Z/yUIhddj8mAZzf4stjasERLJFs+yXaNIm5W4iDZUqf8GTulNsEcsYUCzN9NtkExrY+lzAaXNJC2+SQW63ta2dVPMNeTa2mCxj8ZJ+tkCUM4tifWX3/D+otfShGCYAAA==", "ablation_2ec.csv": "..."}}'''

P = json.loads(PAYLOAD)
os.makedirs('ext/src', exist_ok=True)
os.makedirs('ext/results', exist_ok=True)
for n, b in P['src'].items():
    open('ext/src/' + n, 'wb').write(gzip.decompress(base64.b64decode(b)))
for n, b in P['data'].items():
    open('ext/results/' + n, 'wb').write(gzip.decompress(base64.b64decode(b)))
sys.path.insert(0, 'ext/src')

def load(name):
    d = pd.read_csv('ext/results/' + name)
    if 'family' in d.columns:
        d['family'] = d['family'].fillna('null')   # pandas reads 'null' as NaN
    return d

print('files written to ext/')

---
## A. Verify every reported number from the released result files

### A1. The two-benchmark inversion with 17 methods

Claim: with UniForCE added to the pool, CDK places 9 of 17 on the
clusters-only benchmark (beaten by 7 indices) and 2 of 17 on the
all-blocks benchmark (beaten by none, beating 15 of 16 significantly).

In [ ]:
from scipy.stats import rankdata, wilcoxon

old = load('raw_results.csv')
uf  = load('uniforce_full.csv')
est = old[old.method != 'ORACLE k-means(k*)'].copy()
allr = pd.concat([est, uf], ignore_index=True)
allr['abs_err'] = (allr.k_hat - allr.true_k).abs()

def bench(df, label):
    piv = df.pivot_table(index=['dataset','rep'], columns='method', values='abs_err').dropna()
    R = piv.apply(lambda r: rankdata(r.values), axis=1, result_type='expand'); R.columns = piv.columns
    mr = R.mean().sort_values()
    place = list(mr.index).index('CDK (proposed)') + 1
    ps = {}
    for m in piv.columns:
        if m == 'CDK (proposed)': continue
        d = piv['CDK (proposed)'] - piv[m]
        ps[m] = (wilcoxon(piv['CDK (proposed)'], piv[m])[1] if (d != 0).sum() else 1.0, d.mean())
    items = sorted(ps.items(), key=lambda kv: kv[1][0])
    beat, beats = [], []
    for i, (m, (p, dm)) in enumerate(items):
        if min(1.0, p * (len(items) - i)) < 0.05:
            (beat if dm > 0 else beats).append(m)
    print(f"{label}: CDK mean rank {mr['CDK (proposed)']:.2f}, place {place}/17; "
          f"beaten by {len(beat)}, beats {len(beats)}")
    print('  beaten by:', beat)
    return mr

mr1 = bench(allr[allr.true_k > 1], 'clusters only (780 blocks)')
mr2 = bench(allr, 'all blocks (1010 blocks)')
assert list(mr2.index).index('CDK (proposed)') + 1 == 2

### A2. The certified candidate family changes nothing measurable

Claim: knn + 2-edge-connectivity differs from the conference default on 8
of 1010 blocks, with all headline metrics equal to three decimals.

In [ ]:
k2 = load('knn2ec_full.csv')
cdk = old[old.method == 'CDK (proposed)']
m = cdk.merge(k2, on=['dataset','rep'], suffixes=('_def','_2ec'))
diff = (m.k_hat_def != m.k_hat_2ec).sum()
for tag, d in [('default', cdk), ('knn+2ec', k2)]:
    ae = (d.k_hat - d.true_k).abs().mean()
    ex = (d.k_hat == d.true_k).mean() * 100
    print(f'{tag:8s} |k-k*|={ae:.3f}  exact={ex:.1f}%')
print('blocks where k_hat differs:', diff)
assert diff == 8

### A3. Structureless data, UniForCE, and calibration

Claims: UniForCE returns k=1 on 100% of synthetic nulls (CDK 93.8%);
the per-edge rejection rate at nominal 0.01 is far above nominal in d=2
and near nominal in d=10; every null block with k_hat>1 had at least 9
rejections, so the correct output is carried by graph redundancy.

In [ ]:
nu_uf = uf[(uf.family=='null')]
print('UniForCE null exact: %.1f%%' % ((nu_uf.k_hat==1).mean()*100))
nul = k2[k2.family=='null'].copy()
nul['N'] = (nul.sep_evidence * nul.n_tests).round().astype(int)
nul['d2'] = nul.dataset.str.endswith('_d2')
print(nul.groupby('d2').sep_evidence.mean().rename('per-edge rejection @ nominal 0.01'))
bad = nul[nul.k_hat > 1]
print('null blocks with k_hat>1:', len(bad), ' min rejections among them:', bad.N.min())
assert bad.N.min() >= 9

### A4. The ablation rerun reproduces Table V on the corrected family

In [ ]:
def table5(df):
    df = df.copy(); df['abs_err']=(df.k_hat-df.true_k).abs(); df['exact']=(df.k_hat==df.true_k)
    rows=[]
    for mth, g in df.groupby('method'):
        st=g[g.true_k>1]; nu=g[g.true_k==1]
        rows.append(dict(variant=mth, abs_err=g.abs_err.mean(),
                         exact_struct=st.exact.mean()*100, null=nu.exact.mean()*100))
    return pd.DataFrame(rows).sort_values('abs_err').round(3)

ab_old = load('raw_ablation.csv'); ab_old = ab_old[ab_old.method.str.startswith('CDK')]
ab_new = load('ablation_2ec.csv')
print('original (conference):'); print(table5(ab_old).to_string(index=False))
print(); print('rerun (knn+2ec base):'); print(table5(ab_new).to_string(index=False))

### A5. Contamination (trimmed test) and the hull reference

In [ ]:
rb = load('robust_sweep.csv'); rb['abs_err']=(rb.k_hat-rb.true_k).abs()
print('gamma sweep, contaminated family (true k=4), mean |k-k*|:')
print(rb[rb.family=='noise'].groupby('gamma').abs_err.mean().round(3))
print('gamma sweep, nulls (exact recovery of k=1):')
print(rb[rb.family=='null'].groupby('gamma').apply(lambda g:(g.k_hat==1).mean()).round(3))
h = load('hull_sweep.csv'); h['abs_err']=(h.k_hat-h.true_k).abs()
print('hull reference, structured subset mean |k-k*|: %.3f (box on same blocks: 0.667)'
      % h[h.family!='null'].abs_err.mean())

### A6. The one-sided coverage of k_hat as a lower bound

In [ ]:
cov = (cdk.k_hat <= cdk.true_k).mean()
print('P(k_hat <= k*) over 1010 blocks: %.3f' % cov)
over = cdk[cdk.k_hat > cdk.true_k]
print(over.groupby('family').size().rename('overestimating blocks'))

---
## B. Fresh runs (verify behaviour live)

Each cell below runs an algorithm from scratch on demonstration data with
fixed seeds. Numbers vary slightly with library versions; the qualitative
behaviour is the claim being verified.

In [ ]:
#@title B1. CDK with the certified family: nulls vs clusters (~1 min)
import time
from cdk_geom import CDKGeo2
from datasets import make
cases = [('pure Gaussian noise d2', ('null','x',dict(d=2,kind='gaussian',n=600)), 1),
         ('uniform noise d10',      ('null','x',dict(d=10,kind='uniform',n=600)), 1),
         ('5 clusters d10',         ('sep','x', dict(k=5, d=10,sep=8.0,n=800)), 5),
         ('20 clusters d10',        ('sep','x', dict(k=20,d=10,sep=8.0,n=800)), 20)]
for label, spec, truth in cases:
    X, y, tk = make(spec, 0)
    t0=time.time(); r = CDKGeo2(family='knn', random_state=0).fit(X)
    print(f'{label:24s} true k={truth:2d}   k_hat={r.k:2d}   ({time.time()-t0:.1f}s)')
print()
print('Note: the d=2 null is the fragile case identified in Section A3;')
print('single seeds there can return 2. The full-protocol recovery of k=1')
print('is 93.8% (Section A2/A3), and the paper reports the mechanism.')

In [ ]:
#@title B2. The trimmed test on contaminated data (~1 min)
from cdk_robust import CDKRobust
from datasets import SPECS
noise_spec = [s for s in SPECS if s[1]=='noise_50'][0]
X, y, tk = make(noise_spec, 0)
for g in (0.0, 0.2):
    r = CDKRobust(gamma=g, random_state=0).fit(X)
    print(f'noise_50 (true k={tk}) gamma={g}: k_hat={r.k}')

In [ ]:
#@title B3. UniForCE baseline (~30 s)
from uniforce import uniforce
for label, spec, truth in cases:
    X, y, tk = make(spec, 0)
    k, _ = uniforce(X, random_state=0)
    print(f'{label:24s} true k={truth:2d}   UniForCE k={k:2d}')

In [ ]:
#@title B4. The graph invariants of the geometric families (~5 s)
from scipy.spatial import Delaunay
from cdk_geom import gabriel_edges, rng_edges, _bridges, two_edge_connect
rng_np = np.random.default_rng(0)
C = rng_np.uniform(0,1,(40,2))
tri = Delaunay(C); de=set()
for s in tri.simplices:
    for a,b in ((s[0],s[1]),(s[1],s[2]),(s[0],s[2])): de.add((min(a,b),max(a,b)))
G, R = set(gabriel_edges(C)), set(rng_edges(C))
assert R <= G <= de, 'RNG <= Gabriel <= Delaunay violated'
T = two_edge_connect(C, sorted(G))
assert not _bridges(40, T), 'bridges remain after augmentation'
print(f'RNG {len(R)} <= Gabriel {len(G)} <= Delaunay {len(de)}; no bridges after 2EC. All invariants hold.')

In [ ]:
#@title B5 (optional). Wu-Bien-Panigrahi via the authors' code (~3-8 min)
# Clones the authors' repository and runs one small example.
!git clone -q --depth 1 https://github.com/judywu4800/SI_HierarchicalClustering.git wbp_repo
import sys; sys.path.insert(0,'wbp_repo/src')
from find_best_K import find_best_K_F, generate_alpha_list_exp
from sklearn.datasets import make_blobs
X,_ = make_blobs(n_samples=60, centers=3, n_features=2, cluster_std=1.0,
                 center_box=(-15,15), random_state=0)
al = generate_alpha_list_exp(n=60, total_alpha=0.05)
K_hat, pv, aseq, lab = find_best_K_F(X, tau=0.1, alpha_list=al, linkage='complete', seed=0)
print('WBP on 3 well-separated clusters (n=60): K_hat =', K_hat)

---
Every number in Section A is recomputed from the released result files,
and Section B re-runs the algorithms from their released source. The full
1010-block sweeps take several CPU-hours and are reproduced by
`run_experiments.py` and the runner scripts released with the paper.